# 🛡️ 42 · Trustworthy biological ML: leakage, shift, calibration and conformal prediction
### How to evaluate biological models so that published numbers mean what they claim — donor, homology, preprocessing and temporal leakage; covariate and label shift; calibration; ensembles and the two kinds of uncertainty; conformal prediction with a finite-sample guarantee; interpretability pitfalls; fairness across ancestries; and reporting standards

**BiologyToML** · Level 2 · Research frontier · Part I — Causality, reliability and AI-driven discovery · Module 42 of 44

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/42_trustworthy_ml.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1–2 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy, pandas, scikit-learn, PyTorch (CPU), Matplotlib |
| **Prerequisites** | Level 1 — essential: [Module 23](23_model_selection.ipynb) (cross-validation, selection bias) and [Module 09](09_logistic_regression.ipynb); helpful: Modules 10 (random forests), 21 (posterior uncertainty) and 22 (PyTorch); from Level 2, Modules 30 (single-cell RNA-seq) and 40 (molecular ML) give useful context |
| **Estimated time** | 190 min (reading + running + the ✏️ exercises) |
| **Builds on** | [09 · Logistic regression](09_logistic_regression.ipynb), [10 · Decision trees and random forests](10_decision_trees_random_forests.ipynb), [21 · Bayesian learning and MCMC](21_bayesian_learning_mcmc.ipynb), [23 · Model selection](23_model_selection.ipynb), [30 · Single-cell RNA-seq](30_single_cell_rnaseq.ipynb), [40 · Molecular ML](40_molecular_ml_drug_discovery.ipynb) |
| **Research frontier covered** | data leakage and proper splits, distribution shift, calibration, uncertainty quantification, conformal prediction, interpretability pitfalls, fairness across ancestries, reporting standards (DOME) — see [`FRONTIER_MAP.md`](../FRONTIER_MAP.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> All data are **SIMULATED**, so every performance estimate can be compared with the truth: cells with known type, donor,
> batch and condition (`simulate_scrna`); regulatory-sequence families with known relatedness and a noise-free activity
> (`simulate_regulatory_sequences`); molecules with an oracle potency plus assay noise of known size (`simulate_molecules`);
> two populations' genotypes (`simulate_genotypes`); small in-notebook simulations of hospitals and of drift. "Truth" means
> performance on a **fresh simulated cohort** that no analysis step has touched. In real data relatedness, batches and
> shifts are partly unknown — Section 10 explains what changes.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** with real examples (hospital shortcuts, label-choice bias) why a high test score can fail to transfer, and **use** the DOME checklist to review a study.
2. **Detect and remove** donor, homology, preprocessing and temporal leakage with splits that match the claim, and **verify** each estimate against a fresh simulated cohort.
3. **Distinguish** covariate shift, label shift and concept drift, **detect** shift with a domain classifier, and **correct** a prevalence change with EM.
4. **Measure** calibration (reliability diagrams, ECE), **repair** it with temperature and Platt scaling, and **show** that it degrades under shift.
5. **Decompose** predictive uncertainty into aleatoric and epistemic parts with a deep ensemble and **test** whether it rises out of distribution.
6. **Implement** split-conformal intervals and sets, **verify** their coverage over hundreds of splits, **find** where it fails and **repair** covariate shift with weighted conformal prediction.
7. **Recognise** interpretability pitfalls: importance shared by correlated genes, saliency maps that fail a randomisation sanity check.
8. **Audit** a model across populations, **explain** disparities and what helps, and **write** a model card.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, PyTorch, course toolkit | 5 min |
| 1 · Why trust fails | a hospital shortcut; label choice; the DOME checklist | 15 min |
| 2 · 🧪 Leakage: four ways to fool yourself | donor, homology, preprocessing and temporal leakage vs a fresh cohort | 30 min |
| 3 · Distribution shift | covariate vs label shift; domain classifier; EM prevalence correction | 20 min |
| 4 · Calibration | reliability diagrams, ECE, temperature and Platt scaling, calibration under shift | 20 min |
| 5 · Uncertainty | deep ensembles; aleatoric vs epistemic; does uncertainty rise out of distribution? | 20 min |
| 6 · 🧪 Conformal prediction | split conformal, coverage over 500 splits, subgroups, weighted conformal, cell-type sets | 30 min |
| 7 · Interpretability pitfalls | correlated features in random forests; saliency sanity checks | 15 min |
| 8 · Fairness and equity | a biomarker that behaves differently in two populations; what helps | 15 min |
| 9 · Reproducibility and reporting | seeds, versions, a model card, DOME for an earlier module | 10 min |
| 10 · 🧭 From toy to real data | real resources, tools and the first things that go wrong | 5 min |
| ✅ / 🧠 / 🚀 / 📝 | takeaways, checkpoint, discussion, going further, lab note | 5 min |

## 🧫 The biological question

In 2018 a team trained deep networks to detect pneumonia on chest X-rays from several hospital systems. On held-out images
from the same hospitals they looked excellent; on images from *other* hospital systems performance was often significantly
worse (Zech *et al.* 2018). The networks could tell **which hospital** an image came from, and the hospitals differed in how
many patients had pneumonia: "which hospital?" was a **shortcut** to "how likely is pneumonia?", and shortcuts do not
travel. Obermeyer *et al.* (2019) found that a widely used algorithm for allocating extra care used past health-care *costs*
as its label for health *need*; because less had been spent on Black patients with the same needs, Black patients with the
same risk score were considerably sicker. No train/test split could have revealed that: the label itself was biased.

Biology has its own versions. Protein–protein interaction predictors look far better when test pairs share proteins with
training pairs (Park & Marcotte 2012); genomic models are tested on near-copies of training sequences, on cells of donors
seen in training, or after selecting features on all the data (Whalen *et al.* 2022). A survey found **data leakage** in
hundreds of papers across 17 fields (Kapoor & Narayanan 2023). The DOME recommendations (Walsh *et al.* 2021) ask biological
ML papers to report how the **D**ata were split, how the model was **O**ptimised, what the **M**odel is and how it was
**E**valuated.

This notebook rebuilds each failure on purpose with **SIMULATED** data whose truth we know, then builds the remedies:
splits that match the claim, shift detection and correction, calibration, ensembles, and **conformal prediction** — a
simple recipe that turns any model into one that outputs intervals or sets with a guaranteed coverage rate.

**How do we evaluate biological ML so that published numbers mean what they claim — and attach honest uncertainty to
every prediction?**

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this notebook uses PyTorch (preinstalled on Colab); a CPU is enough.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn"), ("torch", "torch")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import copy
import time
from itertools import product

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import stats
from scipy.optimize import minimize_scalar
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import adjusted_rand_score, roc_auc_score
from sklearn.model_selection import GroupKFold, KFold, StratifiedGroupKFold, cross_val_predict
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

In [ ]:
# ── 0.1 PyTorch configuration ──
torch.manual_seed(SEED)  # PyTorch has its own random-number generator
torch.set_num_threads(1)  # the networks here are tiny: one thread avoids overhead (and is polite on shared CPUs)
OKABE_ITO = ["#0072B2", "#D55E00", "#009E73", "#E69F00", "#CC79A7", "#56B4E9", "#000000"]  # colour-blind-safe palette
T_START = time.time()  # to report the total run time at the end
print("PyTorch", torch.__version__, "| threads:", torch.get_num_threads(), "| device: cpu")

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
We use four simulators — single cells (`simulate_scrna`), regulatory DNA with a noise-free activity oracle (`simulate_regulatory_sequences`, `regulatory_activity`), molecules with an oracle potency (`simulate_molecules`) and genotypes of two populations (`simulate_genotypes`) — plus a few helpers. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def simulate_genotypes(rng: np.random.Generator, n_samples: int, n_snps: int, n_pops: int = 1,
                       fst: float = 0.0, maf_range=(0.05, 0.5)):
    """Simulate 0/1/2 genotypes under Hardy–Weinberg equilibrium.

    Each SNP gets an ancestral allele frequency drawn uniformly from ``maf_range``.  With
    ``n_pops > 1`` and ``fst > 0`` every population's frequency is drawn from the
    Balding–Nichols model Beta(p(1−F)/F, (1−p)(1−F)/F).  Samples are split evenly across
    populations.  Returns ``(G, pop, freqs)``: the N × J genotype matrix, the population of
    each sample and the n_pops × J matrix of population allele frequencies.

    ``G`` is stored as int8 to save memory: convert with ``G.astype(float)`` before arithmetic
    whose intermediate results can exceed 127.  A rare SNP can come out monomorphic (no
    variation) in a small sample — skip such columns in per-SNP tests.
    """
    p = rng.uniform(*maf_range, n_snps)
    if n_pops > 1 and fst > 0:
        a, b = p * (1 - fst) / fst, (1 - p) * (1 - fst) / fst
        freqs = np.clip(rng.beta(a, b, size=(n_pops, n_snps)), 0.001, 0.999)
    else:
        freqs = np.tile(p, (n_pops, 1))
    pop = np.repeat(np.arange(n_pops), int(np.ceil(n_samples / n_pops)))[:n_samples]
    G = rng.binomial(2, freqs[pop]).astype(np.int8)
    return G, pop, freqs


MITO_GENES = ["MT-ND1", "MT-ND2", "MT-CO1", "MT-CO2", "MT-ATP8", "MT-ATP6", "MT-CO3", "MT-ND3",
              "MT-ND4L", "MT-ND4", "MT-ND5", "MT-ND6", "MT-CYB"]


INTERFERON_GENES = ["ISG15", "IFI6", "IFIT1", "IFIT3", "MX1", "OAS1", "IFI44L", "RSAD2", "IRF7", "STAT1"]


CYCLING_GENES = ["MKI67", "TOP2A", "CCNB1", "CDK1", "UBE2C", "BIRC5", "TYMS", "PCNA"]


PBMC_TYPES = {
    "CD4 T": ("lymphoid", ["CD3E", "CD3D", "IL7R", "CD4"], 0.5),
    "CD8 T": ("lymphoid", ["CD3E", "CD3D", "CD8A", "CD8B"], 0.5),
    "NK": ("lymphoid", ["NKG7", "GNLY", "KLRD1", "PRF1"], 2.2),
    "B": ("lymphoid", ["MS4A1", "CD79A", "CD79B", "CD19"], 2.4),
    "Monocyte": ("myeloid", ["CD14", "LYZ", "S100A8", "S100A9"], 2.6),
    "Dendritic": ("myeloid", ["FCER1A", "CST3", "CLEC10A", "HLA-DQA1"], 2.2),
}


def sample_negative_binomial(rng: np.random.Generator, mu, theta):
    """Negative-binomial counts with mean ``mu`` and inverse dispersion ``theta``.

    Variance = mu + mu²/theta (large theta → Poisson).  Drawn as a Gamma–Poisson mixture:
    each count is Poisson with a rate that itself varies from cell to cell.
    """
    mu = np.asarray(mu, float)
    theta = np.broadcast_to(np.asarray(theta, float), mu.shape)
    return rng.poisson(rng.gamma(theta, np.where(mu > 0, mu, 0.0) / theta)).astype(np.int32)


def make_expression_programs(rng: np.random.Generator, n_genes: int, cell_types: dict,
                             n_extra_markers: int = 20, extra_names=()):
    """Log-scale expression profiles (one row per cell type) for a simulated tissue.

    ``cell_types`` maps a type name to (lineage, named marker genes, marker strength).  Every
    type gets its named markers plus ``n_extra_markers`` anonymous ones; types of the same
    lineage share 20 lineage genes.  ``extra_names`` are reserved gene names (placed after the
    markers) that the caller gives its own behaviour.  Returns ``(log_profiles, genes, marker_of)``.
    """
    named = []
    for _, markers, _ in cell_types.values():
        named += [g for g in markers if g not in named]
    reserved = named + [g for g in extra_names if g not in named]
    if len(reserved) > n_genes:
        raise ValueError("n_genes is too small for the requested marker genes")
    genes = np.array(reserved + [f"Gene{j:04d}" for j in range(1, n_genes - len(reserved) + 1)], dtype=object)
    free = np.arange(len(reserved), n_genes)                       # anonymous genes available
    base = rng.normal(0.0, 2.0, n_genes)                           # gene-specific baseline (log scale)
    base[:len(named)] -= 2.0                                       # named markers are low elsewhere
    profiles = np.tile(base, (len(cell_types), 1)) + rng.normal(0, 0.15, (len(cell_types), n_genes))
    marker_of = np.array([""] * n_genes, dtype=object)
    lineages = sorted({v[0] for v in cell_types.values()})
    lineage_genes = {lin: rng.choice(free, 20, replace=False) for lin in lineages}
    for t, (name, (lineage, markers, strength)) in enumerate(cell_types.items()):
        profiles[t, lineage_genes[lineage]] += rng.normal(1.2, 0.3, 20)
        idx = [int(np.flatnonzero(genes == g)[0]) for g in markers]
        profiles[t, idx] += strength + 2.0 + rng.normal(0, 0.2, len(idx))
        extra = rng.choice(free, n_extra_markers, replace=False)
        profiles[t, extra] += rng.normal(0.6 + strength / 2, 0.3, n_extra_markers)
        for j in list(idx) + list(extra):
            marker_of[j] = name if marker_of[j] == "" else marker_of[j] + "/" + name
    return profiles, genes, marker_of


def simulate_scrna(rng: np.random.Generator, n_cells: int = 3000, n_genes: int = 1000, cell_types=None,
                   proportions=None, n_donors: int = 6, n_batches: int = 2, confounded: bool = False,
                   depth: float = 1500, batch_effect: float = 0.25, donor_effect: float = 0.15,
                   stim_effect: float = 1.0, theta: float = 8.0, doublet_rate: float = 0.0,
                   damaged_rate: float = 0.0, cycling_rate: float = 0.0, drop_type_in_batch=None) -> dict:
    """Simulate single-cell RNA-seq UMI counts for a PBMC-like sample collection.

    Design: ``n_donors`` donors, the first half *control* and the second half *stimulated*
    (an interferon response in every cell type plus a monocyte-specific response).  Donor
    ``d`` is sequenced in batch ``d % n_batches`` — or, with ``confounded=True``, every control
    donor in batch 1 and every stimulated donor in batch 2 (a design flaw to learn from).
    Counts are negative binomial around ``depth`` UMIs per cell.  Optional artefacts for
    quality control: doublets, damaged cells (high mitochondrial fraction), cycling cells.
    ``drop_type_in_batch=("B", 1)`` removes B cells from batch 1 (index 1, i.e. "batch2").

    Expression probabilities are renormalised within each cell (a fixed-size library, as in
    sequencing), so in stimulated cells the genes with ``true_logfc == 0`` lose a few per cent of
    their share (about −0.04 on the natural-log scale) to the induced genes: a compositional
    effect that is also present in real data.

    Returns a dict with ``counts`` (cells × genes, int32), ``obs`` (per-cell table with the
    true cell type, donor, batch, condition, …), ``var`` (per-gene table), ``true_logfc``
    (cell type × gene stimulation effect, natural log) and ``log_profiles``.
    """
    cell_types = PBMC_TYPES if cell_types is None else cell_types
    names = list(cell_types)
    T = len(names)
    p = np.full(T, 1 / T) if proportions is None else np.asarray(proportions, float) / np.sum(proportions)
    extra = MITO_GENES + INTERFERON_GENES + CYCLING_GENES
    profiles, genes, marker_of = make_expression_programs(rng, n_genes, cell_types, extra_names=extra)
    G = len(genes)
    is_mito, is_ifn, is_cyc = (np.isin(genes, MITO_GENES), np.isin(genes, INTERFERON_GENES),
                               np.isin(genes, CYCLING_GENES))
    profiles[:, is_mito] = rng.normal(3.0, 0.3, is_mito.sum())          # ≈ 4 % of UMIs are mitochondrial
    profiles[:, is_ifn | is_cyc] = rng.normal(-1.0, 0.4, (T, (is_ifn | is_cyc).sum()))
    # stimulation: interferon genes up everywhere + 40 monocyte-specific responders (up or down)
    true_logfc = np.zeros((T, G))
    true_logfc[:, is_ifn] = stim_effect * rng.normal(1.6, 0.3, is_ifn.sum())
    if "Monocyte" in names:
        responders = rng.choice(np.flatnonzero((marker_of == "") & ~is_mito & ~is_ifn & ~is_cyc), 40, replace=False)
        true_logfc[names.index("Monocyte"), responders] = stim_effect * rng.choice([-1, 1], 40) * rng.normal(1.2, 0.3, 40)
    # donors, batches, conditions
    donor = rng.integers(0, n_donors, n_cells)
    stimulated = donor >= n_donors // 2
    batch = stimulated.astype(int) if confounded else donor % n_batches
    batch = np.minimum(batch, n_batches - 1)
    cell_type = rng.choice(T, n_cells, p=p)
    if drop_type_in_batch is not None:
        t_drop, b_drop = names.index(drop_type_in_batch[0]), drop_type_in_batch[1]
        bad = (cell_type == t_drop) & (batch == b_drop)
        cell_type[bad] = rng.choice(np.delete(np.arange(T), t_drop), bad.sum(), p=np.delete(p, t_drop) / (1 - p[t_drop]))
    batch_shift = rng.normal(0, batch_effect, (n_batches, G))
    batch_depth = np.exp(rng.normal(0, 0.3, n_batches))
    donor_shift = rng.normal(0, donor_effect, (n_donors, G))
    log_rate = (profiles[cell_type] + batch_shift[batch] + donor_shift[donor]
                + true_logfc[cell_type] * stimulated[:, None])
    cycling = rng.random(n_cells) < cycling_rate
    log_rate[np.ix_(cycling, np.flatnonzero(is_cyc))] += 3.0
    damaged = rng.random(n_cells) < damaged_rate
    log_rate[np.ix_(damaged, np.flatnonzero(is_mito))] += 2.2
    lib = depth * batch_depth[batch] * np.exp(rng.normal(0, 0.35, n_cells))
    lib[damaged] *= 0.35
    prob = np.exp(log_rate - log_rate.max(axis=1, keepdims=True))
    prob /= prob.sum(axis=1, keepdims=True)
    theta_g = theta * np.exp(rng.normal(0, 0.5, G))
    mu = lib[:, None] * prob
    doublet = rng.random(n_cells) < doublet_rate
    partner = rng.integers(0, n_cells, n_cells)
    mu[doublet] = 0.9 * mu[doublet] + 0.9 * mu[partner[doublet]]       # two cells in one droplet
    counts = sample_negative_binomial(rng, mu, theta_g)
    obs = pd.DataFrame({
        "cell_type": np.where(doublet, "Doublet", np.array(names, dtype=object)[cell_type]),
        "true_type": np.array(names, dtype=object)[cell_type],
        "lineage": np.array([cell_types[n][0] for n in names], dtype=object)[cell_type],
        "donor": [f"donor{d + 1}" for d in donor],
        "condition": np.where(stimulated, "stimulated", "control"),
        "batch": [f"batch{b + 1}" for b in batch],
        "n_counts": counts.sum(axis=1),
        "n_genes": (counts > 0).sum(axis=1),
        "pct_mito": 100 * counts[:, is_mito].sum(axis=1) / np.maximum(counts.sum(axis=1), 1),
        "is_doublet": doublet, "is_damaged": damaged, "is_cycling": cycling,
    }, index=[f"cell{i:05d}" for i in range(n_cells)])
    var = pd.DataFrame({"marker_of": marker_of, "is_mito": is_mito, "is_interferon": is_ifn,
                        "is_cycling": is_cyc, "theta": theta_g}, index=genes)
    return dict(counts=counts, obs=obs, var=var, log_profiles=pd.DataFrame(profiles, index=names, columns=genes),
                true_logfc=pd.DataFrame(true_logfc, index=names, columns=genes))


IUPAC = {"A": "A", "C": "C", "G": "G", "T": "T", "R": "AG", "Y": "CT", "S": "CG", "W": "AT", "K": "GT",
         "M": "AC", "N": "ACGT"}


REGULATORY_MOTIFS = {
    "AP-1": ("TGASTCA", 1.0),
    "GATA": ("WGATAR", 0.7),
    "SP1": ("GGGGCGGGG", 0.5),
    "E-box": ("CACGTG", 0.8),
    "REST": ("TTCAGCACC", -1.5),        # repressor
}


def reverse_complement(seq: str) -> str:
    """Reverse complement of a DNA string (IUPAC codes are complemented too)."""
    comp = {"A": "T", "C": "G", "G": "C", "T": "A", "R": "Y", "Y": "R", "S": "S", "W": "W", "K": "M",
            "M": "K", "N": "N"}
    return "".join(comp[b] for b in reversed(seq.upper()))


def find_motif_sites(seq: str, consensus: str) -> list:
    """Start positions where ``consensus`` (IUPAC) matches ``seq`` on either strand (overlaps allowed)."""
    import re
    sites = set()
    for c in {consensus, reverse_complement(consensus)}:
        pattern = "(?=" + "".join(f"[{IUPAC[b]}]" for b in c) + ")"
        sites.update(m.start() for m in re.finditer(pattern, seq))
    return sorted(sites)


def regulatory_activity(seq: str) -> float:
    """Noise-free enhancer activity of a DNA sequence under the course's motif grammar.

    Activity = Σ motif effects (sites in the central half count fully, flanking sites 60 %)
    + 1.0 if an AP-1 and a GATA site lie within 30 bp of each other (cooperative binding)
    + 2·(GC fraction − 0.5).  This is the "true biology" that Module 36's networks must learn.
    """
    L = len(seq)
    total, starts = 0.0, {}
    for name, (consensus, effect) in REGULATORY_MOTIFS.items():
        starts[name] = find_motif_sites(seq, consensus)
        for s in starts[name]:
            total += effect * (1.0 if L / 4 <= s + len(consensus) / 2 <= 3 * L / 4 else 0.6)
    if any(abs(a - g) <= 30 for a in starts["AP-1"] for g in starts["GATA"]):
        total += 1.0
    gc = (seq.count("G") + seq.count("C")) / max(L, 1)
    return total + 2.0 * (gc - 0.5)


def simulate_regulatory_sequences(rng: np.random.Generator, n_seqs: int = 4000, length: int = 200,
                                  motif_rate: float = 1.5, noise: float = 0.3, n_chromosomes: int = 8) -> dict:
    """Simulate an enhancer reporter assay (MPRA / STARR-seq style): DNA sequence → activity.

    Background DNA has a per-sequence GC content; each sequence receives Poisson(``motif_rate``)
    planted motif instances at random positions and orientations (sampled from the IUPAC
    consensus).  The measured activity is ``regulatory_activity(seq)`` plus Gaussian noise,
    so motifs that occur by chance count too, exactly as in real DNA.  Each sequence is
    assigned to a chromosome so that models can be tested on held-out chromosomes.

    Returns a dict with ``sequences`` (list of str), ``activity``, ``label`` (1 = active, top
    40 %), ``chromosome`` and ``sites`` (DataFrame of planted motifs).
    """
    names = list(REGULATORY_MOTIFS)
    seqs, rows = [], []
    for i in range(n_seqs):
        gc = rng.beta(8, 8)
        s = list(rng.choice(list("ACGT"), length, p=[(1 - gc) / 2, gc / 2, gc / 2, (1 - gc) / 2]))
        for _ in range(rng.poisson(motif_rate)):
            m = names[rng.integers(len(names))]
            inst = "".join(rng.choice(list(IUPAC[b])) for b in REGULATORY_MOTIFS[m][0])
            strand = "+" if rng.random() < 0.5 else "-"
            if strand == "-":
                inst = reverse_complement(inst)
            start = int(rng.integers(0, length - len(inst)))
            s[start:start + len(inst)] = list(inst)
            rows.append((i, m, start, strand))
        seqs.append("".join(s))
    activity = np.array([regulatory_activity(s) for s in seqs]) + rng.normal(0, noise, n_seqs)
    label = (activity > np.quantile(activity, 0.6)).astype(int)
    chromosome = np.array([f"chr{c + 1}" for c in rng.integers(0, n_chromosomes, n_seqs)], dtype=object)
    sites = pd.DataFrame(rows, columns=["seq_id", "motif", "start", "strand"])
    return dict(sequences=seqs, activity=activity, label=label, chromosome=chromosome, sites=sites)


_AR = 1.5


MOLECULE_SCAFFOLDS = {
    "benzene": ([("C", 1)] * 6, [(i, (i + 1) % 6, _AR) for i in range(6)], [0, 1, 2, 3]),
    "pyridine": ([("N", 1)] + [("C", 1)] * 5, [(i, (i + 1) % 6, _AR) for i in range(6)], [2, 3, 4]),
    "pyrimidine": ([("N", 1), ("C", 1), ("N", 1), ("C", 1), ("C", 1), ("C", 1)],
                   [(i, (i + 1) % 6, _AR) for i in range(6)], [1, 3, 4]),
    "thiophene": ([("S", 1)] + [("C", 1)] * 4, [(i, (i + 1) % 5, _AR) for i in range(5)], [1, 2, 4]),
    "furan": ([("O", 1)] + [("C", 1)] * 4, [(i, (i + 1) % 5, _AR) for i in range(5)], [1, 2, 4]),
    "cyclohexane": ([("C", 0)] * 6, [(i, (i + 1) % 6, 1) for i in range(6)], [0, 1, 2, 3]),
    "piperidine": ([("N", 0)] + [("C", 0)] * 5, [(i, (i + 1) % 6, 1) for i in range(6)], [0, 2, 3]),
    "morpholine": ([("O", 0), ("C", 0), ("C", 0), ("N", 0), ("C", 0), ("C", 0)],
                   [(i, (i + 1) % 6, 1) for i in range(6)], [3, 1, 2]),
    "naphthalene": ([("C", 1)] * 10, [(0, 1, _AR), (1, 2, _AR), (2, 3, _AR), (3, 4, _AR), (4, 5, _AR), (5, 0, _AR),
                                      (4, 6, _AR), (6, 7, _AR), (7, 8, _AR), (8, 9, _AR), (9, 5, _AR)], [0, 1, 2, 7]),
    "quinoline": ([("N", 1)] + [("C", 1)] * 9, [(0, 1, _AR), (1, 2, _AR), (2, 3, _AR), (3, 4, _AR), (4, 5, _AR),
                                                (5, 0, _AR), (4, 6, _AR), (6, 7, _AR), (7, 8, _AR), (8, 9, _AR),
                                                (9, 5, _AR)], [1, 2, 7]),
}


MOLECULE_SUBSTITUENTS = {      # attachment atom = index 0
    "methyl": ([("C", 0)], []),
    "ethyl": ([("C", 0), ("C", 0)], [(0, 1, 1)]),
    "hydroxyl": ([("O", 0)], []),
    "amino": ([("N", 0)], []),
    "fluoro": ([("F", 0)], []),
    "chloro": ([("Cl", 0)], []),
    "methoxy": ([("O", 0), ("C", 0)], [(0, 1, 1)]),
    "carboxyl": ([("C", 0), ("O", 0), ("O", 0)], [(0, 1, 2), (0, 2, 1)]),
    "amide": ([("C", 0), ("O", 0), ("N", 0)], [(0, 1, 2), (0, 2, 1)]),
    "nitrile": ([("C", 0), ("N", 0)], [(0, 1, 3)]),
    "trifluoromethyl": ([("C", 0), ("F", 0), ("F", 0), ("F", 0)], [(0, 1, 1), (0, 2, 1), (0, 3, 1)]),
    "sulfonamide": ([("S", 0), ("O", 0), ("O", 0), ("N", 0)], [(0, 1, 2), (0, 2, 2), (0, 3, 1)]),
    "dimethylamino": ([("N", 0), ("C", 0), ("C", 0)], [(0, 1, 1), (0, 2, 1)]),
    "acetyl": ([("C", 0), ("O", 0), ("C", 0)], [(0, 1, 2), (0, 2, 1)]),
    "phenyl": ([("C", 1)] * 6, [(i, (i + 1) % 6, _AR) for i in range(6)]),
}


_VALENCE = {"C": 4, "N": 3, "O": 2, "S": 2, "F": 1, "Cl": 1}


_MASS = {"C": 12.011, "N": 14.007, "O": 15.999, "S": 32.06, "F": 18.998, "Cl": 35.45, "H": 1.008}


def molecule_to_smiles(atoms, bonds) -> str:
    """Write a molecular graph as a SMILES string (depth-first, ring closures as digits; not canonical)."""
    n = len(atoms)
    nbr = {i: [] for i in range(n)}
    for i, j, o in bonds:
        nbr[i].append((j, o))
        nbr[j].append((i, o))
    for i in nbr:
        nbr[i].sort()
    parent, order, closures, seen = {0: None}, [], {}, set()

    def visit(a):                                    # first pass: tree edges vs ring-closure edges
        seen.add(a)
        order.append(a)
        for b, o in nbr[a]:
            if b not in seen:
                parent[b] = a
                visit(b)
            elif b != parent[a] and (min(a, b), max(a, b)) not in closures:
                closures[(min(a, b), max(a, b))] = o
    visit(0)
    ring_at = {i: [] for i in range(n)}
    for k, (pair, o) in enumerate(closures.items()):
        for a in pair:
            ring_at[a].append((k + 1, o, pair))

    def bond_symbol(o, a, b):
        if o == 2:
            return "="
        if o == 3:
            return "#"
        return "-" if (o == 1 and atoms[a][1] and atoms[b][1]) else ""

    def write(a):
        el, arom = atoms[a]
        s = el.lower() if arom else el
        for digit, o, pair in ring_at[a]:
            first = a == min(pair, key=order.index)
            s += (bond_symbol(o, *pair) if first else "") + (str(digit) if digit < 10 else f"%{digit}")
        kids = [(b, o) for b, o in nbr[a] if parent.get(b) == a]
        for idx, (b, o) in enumerate(kids):
            sub = bond_symbol(o, a, b) + write(b)
            s += sub if idx == len(kids) - 1 else f"({sub})"
        return s
    return write(0)


def molecule_descriptors(atoms, bonds) -> dict:
    """Simple descriptors: heavy atoms, implicit H, molecular weight, H-bond donors/acceptors,
    aromatic atoms, rings and a simplified (Crippen-like) octanol–water partition coefficient."""
    used = np.zeros(len(atoms))
    for i, j, o in bonds:
        used[i] += o
        used[j] += o
    n_h = np.array([max(0, int(np.floor(_VALENCE[el] - used[k] + 1e-9))) if el in ("C", "N", "O") else 0
                    for k, (el, _) in enumerate(atoms)])
    els = [el for el, _ in atoms]
    mw = sum(_MASS[e] for e in els) + _MASS["H"] * n_h.sum()
    hbd = sum(1 for k, e in enumerate(els) if e in ("N", "O") and n_h[k] > 0)
    hba = sum(1 for e in els if e in ("N", "O"))
    contrib = {"C": 0.25, "N": -0.75, "O": -0.55, "S": 0.4, "F": 0.35, "Cl": 0.7}
    logp = sum(contrib[e] + (0.12 if a else 0) for e, a in atoms) - 0.3 * hbd
    return dict(heavy_atoms=len(atoms), n_hydrogens=int(n_h.sum()), mol_weight=round(mw, 2), hbd=hbd, hba=hba,
                aromatic_atoms=int(sum(a for _, a in atoms)), rings=len(bonds) - len(atoms) + 1,
                logp=round(logp, 3))


def assemble_molecule(scaffold: str, substituents: dict):
    """Molecular graph (atoms, bonds) of ``scaffold`` carrying ``substituents`` = {site index: substituent name}."""
    atoms, bonds, sites = MOLECULE_SCAFFOLDS[scaffold]
    atoms, bonds = list(atoms), list(bonds)
    for s, name in sorted(substituents.items()):
        f_atoms, f_bonds = MOLECULE_SUBSTITUENTS[name]
        off = len(atoms)
        atoms += f_atoms
        bonds += [(i + off, j + off, o) for i, j, o in f_bonds] + [(sites[s], off, 1)]
    return atoms, bonds


def simulate_molecules(rng: np.random.Generator, n_molecules: int = 2000, noise: float = 0.3) -> dict:
    """Simulate a small-molecule library with a bioactivity (pIC50) and a solubility (logS) assay.

    Each molecule = one of 10 ring scaffolds + 1–3 substituents at the scaffold's attachment
    sites.  Hidden structure–activity rules (the "biology" a model must learn): each scaffold
    has its own baseline potency; a hydrogen-bond donor at site 0 (the key pocket) adds +1.2;
    small lipophilic groups at site 1 add +0.5; bulky groups at site 2 clash (−1.0); very
    lipophilic molecules lose potency (−0.4 per logP unit above 3); plus small random
    site × substituent effects.  Solubility falls with logP and size.

    Returns a dict with ``table`` (SMILES, scaffold, substituents, descriptors, measured
    ``pIC50`` and ``logS``), ``graphs`` (list of (atoms, bonds)), ``scaffold_effect``, and
    ``oracle(scaffold, substituents)`` → noise-free (pIC50, logS) of *any* design, e.g.
    ``oracle("pyridine", {0: "amino", 2: "methyl"})`` — the ground truth for design (Module 39).
    """
    scaffolds, subs = list(MOLECULE_SCAFFOLDS), list(MOLECULE_SUBSTITUENTS)
    weights = rng.dirichlet(np.full(len(scaffolds), 2.0))                 # some scaffolds are rare
    scaffold_effect = dict(zip(scaffolds, rng.normal(5.5, 0.8, len(scaffolds))))
    site_effect = rng.normal(0, 0.2, (4, len(subs)))
    donors = {"hydroxyl", "amino", "amide", "sulfonamide", "carboxyl"}
    lipophilic = {"methyl", "ethyl", "chloro", "trifluoromethyl"}
    bulky = {"phenyl", "sulfonamide", "dimethylamino", "trifluoromethyl"}

    def oracle(scaffold: str, substituents: dict):
        """Noise-free (pIC50, logS) of a scaffold + {site: substituent} design under the hidden rules."""
        desc = molecule_descriptors(*assemble_molecule(scaffold, substituents))
        act = scaffold_effect[scaffold] + sum(site_effect[s, subs.index(nm)] for s, nm in substituents.items())
        act += 1.2 * (substituents.get(0) in donors) + 0.5 * (substituents.get(1) in lipophilic) \
            - 1.0 * (substituents.get(2) in bulky) - 0.4 * max(0.0, desc["logp"] - 3.0)
        logs = 0.5 - 0.8 * desc["logp"] - 0.006 * (desc["mol_weight"] - 150) + 0.2 * desc["hbd"]
        return float(act), float(logs)

    rows, graphs, seen = [], [], set()
    while len(rows) < n_molecules:
        sc = scaffolds[rng.choice(len(scaffolds), p=weights)]
        n_sites = len(MOLECULE_SCAFFOLDS[sc][2])
        k = int(rng.integers(1, min(3, n_sites) + 1))
        chosen = {int(s): subs[rng.integers(len(subs))] for s in sorted(rng.choice(n_sites, k, replace=False))}
        key = (sc, tuple(sorted(chosen.items())))
        if key in seen:
            continue
        seen.add(key)
        atoms, bonds = assemble_molecule(sc, chosen)
        act, logs = oracle(sc, chosen)
        rows.append(dict(smiles=molecule_to_smiles(atoms, bonds), scaffold=sc,
                         substituents="; ".join(f"site{s}:{nm}" for s, nm in sorted(chosen.items())),
                         **molecule_descriptors(atoms, bonds),
                         pIC50=act + rng.normal(0, noise), logS=logs + rng.normal(0, noise)))
        graphs.append((atoms, bonds))
    return dict(table=pd.DataFrame(rows), graphs=graphs, scaffold_effect=scaffold_effect, oracle=oracle)


def sigmoid(z):
    """Logistic function 1 / (1 + e^(−z)), written via tanh so that it never overflows."""
    return 0.5 * (1.0 + np.tanh(0.5 * np.asarray(z, float)))


def one_hot_dna(seq: str) -> np.ndarray:
    """One-hot encode a DNA string as an L × 4 array with columns A, C, G, T (N → all zeros)."""
    lookup = {"A": 0, "C": 1, "G": 2, "T": 3}
    out = np.zeros((len(seq), 4))
    for i, base in enumerate(seq.upper()):
        if base in lookup:
            out[i, lookup[base]] = 1.0
    return out


def log_normalize(counts, target_sum: float = 1e4):
    """Library-size normalisation + log: log(1 + target_sum · count / total count of the cell)."""
    counts = np.asarray(counts, float)
    total = counts.sum(axis=1, keepdims=True)
    return np.log1p(target_sum * counts / np.where(total > 0, total, 1.0))


def pca(X, n_components: int = 2):
    """Principal component analysis by SVD of the column-centred data.

    Returns ``(scores, components, explained_variance_ratio)``: scores are N × k, components k × J.
    """
    X = np.asarray(X, float)
    Xc = X - X.mean(axis=0)
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    var = S ** 2 / (len(X) - 1)
    return U[:, :n_components] * S[:n_components], Vt[:n_components], var[:n_components] / var.sum()


def plot_embedding(Z, labels, ax=None, title: str = "", s: float = 4, legend: bool = True, cmap=None):
    """Scatter a 2-D embedding (PCA, t-SNE, latent space, …) coloured by a categorical or numeric label."""
    Z = np.asarray(Z, float)
    labels = np.asarray(labels)
    ax = ax or plt.gca()
    if labels.dtype.kind in "fc":
        sc = ax.scatter(Z[:, 0], Z[:, 1], c=labels, s=s, cmap=cmap or "viridis", linewidths=0)
        plt.colorbar(sc, ax=ax, shrink=0.8)
    else:
        palette = plt.get_cmap("tab10" if len(np.unique(labels)) <= 10 else "tab20")
        for k, lab in enumerate(np.unique(labels)):
            m = labels == lab
            ax.scatter(Z[m, 0], Z[m, 1], s=s, color=palette(k % palette.N), label=str(lab), linewidths=0)
        if legend:
            ax.legend(markerscale=3, fontsize=7, frameon=False, loc="best")
    ax.set_xticks([]); ax.set_yticks([])
    if title:
        ax.set_title(title)
    return ax


print("🧰 Toolkit loaded:", ", ".join(['simulate_genotypes', 'sample_negative_binomial', 'make_expression_programs', 'simulate_scrna', 'reverse_complement', 'find_motif_sites', 'regulatory_activity', 'simulate_regulatory_sequences', 'molecule_to_smiles', 'molecule_descriptors', 'assemble_molecule', 'simulate_molecules', 'sigmoid', 'one_hot_dna', 'log_normalize', 'pca', 'plot_embedding']))

---
## 1 · Why trust fails: shortcuts, label choice and reporting standards

> 📍 **Where we are.** [Module 23](23_model_selection.ipynb) taught the cardinal rule: estimate performance on data the
> model never saw. This module asks a harder question — *never saw in what sense?* A test set is only useful if it is as
> different from the training set as the data the model will meet in use. We start with the hospital story in miniature.

**A shortcut in three hospitals.** Each SIMULATED patient has 8 blood biomarkers (four of them shifted upwards by disease)
and 2 "site signature" features — think of a scanner's calibration or a lab's processing protocol, which say nothing about
disease but differ between hospitals. Hospital A is a community hospital (10 % of its patients have the disease), hospital
B a referral centre (70 %). We train on A and B pooled and then deploy at hospital C (30 % prevalence, its own site
signature).

In [ ]:
# ── 1.1 A shortcut: the model learns *where* a sample came from (SIMULATED hospitals) ──
def simulate_hospital(rng, n, prevalence, site_shift, effect=0.5):
    """n patients: 8 biomarkers (the first 4 rise by `effect` with disease) + 2 site-signature features."""
    y = (rng.random(n) < prevalence).astype(int)
    biomarkers = rng.normal(size=(n, 8))
    biomarkers[:, :4] += effect * y[:, None]
    site = rng.normal(site_shift, 0.5, size=(n, 2))  # depends on the hospital only, never on disease
    return np.c_[biomarkers, site], y


rng = np.random.default_rng(SEED + 11)
X_a, y_a = simulate_hospital(rng, 1500, prevalence=0.10, site_shift=0.0)  # community hospital
X_b, y_b = simulate_hospital(rng, 1500, prevalence=0.70, site_shift=2.0)  # referral centre
X_c, y_c = simulate_hospital(rng, 2000, prevalence=0.30, site_shift=1.0)  # the next hospital (deployment)
X_ab, y_ab = np.r_[X_a, X_b], np.r_[y_a, y_b]
order = rng.permutation(len(y_ab))
tr_h, te_h = order[:2000], order[2000:]  # internal random split of A + B
rows = []
for name, cols in [("all 10 features", slice(None)), ("biomarkers only", slice(0, 8))]:
    model = LogisticRegression(max_iter=1000).fit(X_ab[tr_h][:, cols], y_ab[tr_h])
    rows.append((name, roc_auc_score(y_ab[te_h], model.predict_proba(X_ab[te_h][:, cols])[:, 1]),
                 roc_auc_score(y_c, model.predict_proba(X_c[:, cols])[:, 1])))
shortcut = pd.DataFrame(rows, columns=["model", "internal AUROC (A+B)", "external AUROC (C)"])
oracle_auc_c = roc_auc_score(y_c, X_c[:, :4].sum(1))  # the true disease signal: sum of the 4 affected biomarkers
print(shortcut.round(3).to_string(index=False))
print(f"oracle AUROC at hospital C (true biomarker signal only): {oracle_auc_c:.3f}")

In [ ]:
# ── 1.2 Why it happens: the site signature predicts disease *in the training hospitals* ──
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.5))
for X_, y_, name, c in [(X_a, y_a, "A (10 % disease)", OKABE_ITO[0]), (X_b, y_b, "B (70 %)", OKABE_ITO[1]),
                        (X_c, y_c, "C (30 %, deployment)", OKABE_ITO[2])]:
    a1.hist(X_[:, 8], bins=40, alpha=0.55, color=c, label=name, density=True)
a1.set_xlabel("site-signature feature 1"); a1.set_ylabel("density"); a1.legend()
a1.set_title("SIMULATED: the site signature identifies the hospital,\nand the hospitals differ in prevalence")
xpos = np.arange(2)
a2.bar(xpos - 0.18, shortcut.iloc[:, 1], 0.36, color=OKABE_ITO[0], label="internal test (A + B)")
a2.bar(xpos + 0.18, shortcut.iloc[:, 2], 0.36, color=OKABE_ITO[1], label="external test (C)")
a2.axhline(oracle_auc_c, color="k", ls="--", lw=1, label="oracle (true signal)")
a2.set_xticks(xpos); a2.set_xticklabels(shortcut["model"]); a2.set_ylim(0.5, 1.0); a2.set_ylabel("AUROC")
a2.set_title("the shortcut inflates the internal score\nand does not travel"); a2.legend(loc="upper right", fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** With all ten features the model reaches an internal AUROC of 0.88 — far above the oracle's
0.75, the best that the disease biology itself allows. The extra comes from the site signature, which in the training
hospitals is a proxy for prevalence ("site B → probably diseased"). At hospital C the proxy points
the wrong way and the AUROC falls to 0.72, *below* the model that never saw the site features (0.74 externally and 0.75
internally). The internal test set was not wrong — it answered a different question ("how well does this work for more
patients from A and B?") from the one that matters ("how well will it work at the next hospital?"). And not every failure
is a split: in the Obermeyer study the model faithfully predicted its label — cost — but cost was a biased proxy for need.

**The DOME checklist** (Walsh *et al.* 2021) lists what a reader needs in order to trust an ML result in biology:

| | Report | Guards against |
|---|---|---|
| **D**ata | provenance, size, balance; **how the data were split** and how independent the splits are | leakage (§2), shift (§3) |
| **O**ptimisation | preprocessing, feature selection, tuning — all **inside** the training data | selection bias (Module 23) |
| **M**odel | what it is, interpretability, availability of code and weights | irreproducibility (§9) |
| **E**valuation | metrics with uncertainty, baselines, a test set that matches the claim, calibration | over-optimism (§4–6) |

> 🔗 **Builds on** [Module 23 · Model selection](23_model_selection.ipynb), Section 6: there the test set had to be untouched
> by *training*; here it must also be untouched by *shortcuts* — features, people, sequences or time periods that link it to
> the training set more closely than deployment data will be.

✏️ **Your turn.** In cell 1.1 give hospital B the prevalence of A (`prevalence=0.10`). Does the all-feature model still beat
the oracle internally? Why is the site signature then harmless — and why would you still leave it out?

---
## 2 · 🧪 Leakage: four ways to fool yourself

> 📍 **Where we are.** Leakage is information in the development data that will not be available when the model is used
> and that makes estimated performance too good. We provoke four kinds — **donor**, **homology**, **preprocessing**,
> **temporal** — and for each compare a naive estimate, an honest estimate and the truth.

The principle: **the test fold must be as independent of the training folds as future data will be of the deployed model.**
For "works for new patients" hold out whole patients; for "new protein families", whole families; for "next year", the future.

🔬 **Assay box — cells, donors and pseudo-replication.** Single-cell studies profile thousands of cells from each of a
handful of donors (Module 30). Cells of one donor share a genotype, a sample-handling history and ambient RNA, so they are
**not independent observations** of a disease: the unit of replication is the donor. Treating cells as replicates is
*pseudo-replication* — the reason for pseudobulk analysis in [Module 24](24_glm_count_data.ipynb).

**(a) Donor leakage.** 20 SIMULATED donors (10 control, 10 interferon-stimulated), a *weak* stimulation effect and realistic
donor variation: can a classifier tell from **one cell** whether it came from a stimulated donor? Six donors (three per
condition) are locked away as a fresh cohort.

In [ ]:
# ── 2.1 Donor leakage: a SIMULATED cohort of 20 donors (weak stimulation, real donor effects) ──
rng = np.random.default_rng(SEED + 21)
cohort = simulate_scrna(rng, n_cells=4000, n_genes=300, n_donors=20, stim_effect=0.7, donor_effect=0.25, depth=1000)
donor = cohort["obs"]["donor"].to_numpy(str)
y_stim = (cohort["obs"]["condition"].to_numpy(str) == "stimulated").astype(int)
X_cells = log_normalize(cohort["counts"])
fresh = np.isin(donor, ["donor1", "donor2", "donor3", "donor11", "donor12", "donor13"])  # locked away until the end
dev = ~fresh


def cv_score(X, y, splitter, groups=None, make_model=None):
    """Mean test-fold accuracy; the model (scaling + classifier) is refitted inside every fold."""
    make_model = make_model or (lambda: make_pipeline(StandardScaler(), LogisticRegression(C=0.01, max_iter=2000)))
    return np.mean([make_model().fit(X[tr], y[tr]).score(X[te], y[te]) for tr, te in splitter.split(X, y, groups)])


t0 = time.time()
acc_cells = cv_score(X_cells[dev], y_stim[dev], KFold(5, shuffle=True, random_state=SEED))
acc_donors = cv_score(X_cells[dev], y_stim[dev], StratifiedGroupKFold(5, shuffle=True, random_state=SEED), donor[dev])
final_model = make_pipeline(StandardScaler(), LogisticRegression(C=0.01, max_iter=2000)).fit(X_cells[dev], y_stim[dev])
acc_fresh = final_model.score(X_cells[fresh], y_stim[fresh])
print(f"SIMULATED: {dev.sum()} development cells from 14 donors, {fresh.sum()} cells from 6 fresh donors")
print(f"accuracy, random split of cells      : {acc_cells:.3f}")
print(f"accuracy, donors held out (grouped)  : {acc_donors:.3f}")
print(f"accuracy on the fresh donors (truth) : {acc_fresh:.3f}   ({time.time() - t0:.1f} s)")

In [ ]:
# ── 2.2 Why: cells carry a donor fingerprint ──
donor_acc = cv_score(X_cells[dev], donor[dev], KFold(5, shuffle=True, random_state=SEED),
                     make_model=lambda: make_pipeline(StandardScaler(), LogisticRegression(C=0.01, max_iter=2000)))
print(f"predicting WHICH of the 14 donors a cell came from: accuracy {donor_acc:.3f} (chance = {1 / 14:.3f})")

🔍 **What just happened?** With a random split of cells the classifier scores 0.91 — but every donor sits on both sides of
the split, and cells carry a **donor fingerprint** (cell 2.2: the donor of a single cell can be named with accuracy 0.95,
chance 0.07). The model scores well by recognising donors it has seen and recalling their condition. Holding out whole donors
(`StratifiedGroupKFold`, which also keeps both conditions in every fold) gives 0.66; the six untouched donors give 0.75. The
two honest numbers differ — with 14 development donors each fold holds out about three, so the estimate is noisy — but both
are far below 0.91, which answers "how well can we classify more cells from *these* donors?".

⚠️ **The same trap, other names:** several biopsies or time points per patient, several images per slide, several mice per
litter. **Ask what the unit of deployment is and split by it.**

**(b) Homology leakage.** Sequences come in families. We simulate 300 ancestral enhancers (`simulate_regulatory_sequences`),
each with 2–8 descendants carrying 4 % point mutations; activity = the noise-free oracle `regulatory_activity` + noise. A
random split puts siblings on both sides. Benchmarks instead **cluster sequences by identity** (CD-HIT, MMseqs2) and hold out
whole clusters; our clustering is not told the families — it must find them.

In [ ]:
# ── 2.3 Homology leakage: SIMULATED families of related enhancer sequences ──
def make_sequence_families(rng, n_families=300, length=150, kids=(2, 8), mut_rate=0.04, noise=0.3):
    """Ancestors from simulate_regulatory_sequences; each has 2–8 descendants with point mutations."""
    ancestors = simulate_regulatory_sequences(rng, n_seqs=n_families, length=length, motif_rate=2.0)["sequences"]
    seqs, family = [], []
    for f, s in enumerate(ancestors):
        for _ in range(rng.integers(kids[0], kids[1] + 1)):
            arr = np.array(list(s))
            hit = rng.random(length) < mut_rate
            arr[hit] = rng.choice(list("ACGT"), hit.sum())
            seqs.append("".join(arr)); family.append(f)
    activity = np.array([regulatory_activity(s) for s in seqs]) + rng.normal(0, noise, len(seqs))
    return seqs, np.array(family), activity


def kmer_counts(seqs, k=4):
    """Count all 4^k k-mers in every sequence (a simple, alignment-free representation)."""
    index = {"".join(p): i for i, p in enumerate(product("ACGT", repeat=k))}
    out = np.zeros((len(seqs), len(index)))
    for n, s in enumerate(seqs):
        for i in range(len(s) - k + 1):
            out[n, index[s[i:i + k]]] += 1
    return out


def identity_clusters(seqs, threshold=0.6):
    """Single-linkage clusters of equal-length sequences joined when their identity ≥ threshold (CD-HIT-like)."""
    oh = np.array([one_hot_dna(s).ravel() for s in seqs])
    identity = oh @ oh.T / len(seqs[0])  # fraction of identical positions for every pair
    _, labels = connected_components(csr_matrix(identity >= threshold), directed=False)
    return labels, identity


rng = np.random.default_rng(SEED + 22)
fam_seqs, family, fam_y = make_sequence_families(rng)
fam_X = kmer_counts(fam_seqs)
clusters, identity = identity_clusters(fam_seqs)
print(f"SIMULATED: {len(fam_seqs)} sequences from {family.max() + 1} families")
print(f"🧪 identity clustering found {clusters.max() + 1} clusters; agreement with the true families "
      f"(adjusted Rand index) = {adjusted_rand_score(family, clusters):.3f}")

In [ ]:
# ── 2.4 Random split vs cluster split vs fresh, unrelated sequences ──
fresh_seq = simulate_regulatory_sequences(np.random.default_rng(SEED + 99), n_seqs=600, length=150, motif_rate=2.0)
fresh_X = kmer_counts(fresh_seq["sequences"])
fresh_y = fresh_seq["activity"]


def r2_score_np(pred, y):
    """Coefficient of determination R² = 1 − SS_res / SS_tot."""
    return 1 - ((y - pred) ** 2).sum() / ((y - y.mean()) ** 2).sum()


rows = []
for name, make in [("ridge on 4-mer counts", lambda: Ridge(alpha=10.0)), ("1-nearest neighbour", lambda: KNeighborsRegressor(1))]:
    row = [name]
    for splitter, groups in [(KFold(5, shuffle=True, random_state=SEED), None), (GroupKFold(5), clusters)]:
        pred = np.zeros(len(fam_y))
        for tr, te in splitter.split(fam_X, fam_y, groups):
            pred[te] = make().fit(fam_X[tr], fam_y[tr]).predict(fam_X[te])
        row.append(r2_score_np(pred, fam_y))
    row.append(r2_score_np(make().fit(fam_X, fam_y).predict(fresh_X), fresh_y))
    rows.append(row)
homology = pd.DataFrame(rows, columns=["model", "R² random split", "R² cluster split", "R² fresh sequences (truth)"])
print(homology.round(3).to_string(index=False))

In [ ]:
# ── 2.5 The diagnostic: how similar is each test sequence to its closest training sequence? ──
tr_r, te_r = next(KFold(5, shuffle=True, random_state=SEED).split(fam_X))
tr_c, te_c = next(GroupKFold(5).split(fam_X, fam_y, clusters))
nn_random = identity[np.ix_(te_r, tr_r)].max(1)
nn_cluster = identity[np.ix_(te_c, tr_c)].max(1)
fig, ax = plt.subplots(figsize=(7, 3.2))
bins = np.linspace(0.25, 1.0, 46)
ax.hist(nn_random, bins=bins, alpha=0.6, color=OKABE_ITO[1], label=f"random split (median {np.median(nn_random):.2f})")
ax.hist(nn_cluster, bins=bins, alpha=0.6, color=OKABE_ITO[0], label=f"cluster split (median {np.median(nn_cluster):.2f})")
ax.set_xlabel("identity of a test sequence to its nearest training sequence"); ax.set_ylabel("test sequences")
ax.set_title("SIMULATED: a random split leaves near-copies of training sequences in the test set"); ax.legend()
plt.tight_layout(); plt.show()

🔍 **What just happened?** Identity clustering recovered the families perfectly (adjusted Rand index 1.000): siblings share
about 95 % of positions, the most similar unrelated sequence only about 37 %. The histogram is the diagnostic every sequence
benchmark should show — after a random split the median test sequence has a 95 %-identical training sibling. Ridge regression
on 4-mer counts scores R² = 0.56 with a random split, 0.23 with a cluster split and 0.28 on genuinely new sequences. A
1-nearest-neighbour "model", which copies the activity of the most similar training sequence, looks almost as good under a
random split (0.51) and is **much worse than predicting the mean** on new sequences (R² = −0.61): it learned family
resemblance, not motif grammar. (The cluster split is slightly pessimistic — it trains on 80 % of the families — but both
honest estimates are far from the leaky one.)

**(c) Preprocessing leakage** is the trap of [Module 23](23_model_selection.ipynb), Section 6: any step that uses the labels —
feature selection, thresholds, tuning — must be repeated inside every training fold. The classic demonstration: 3,000 "genes"
of pure noise and **random** labels, so the true accuracy is exactly 50 %.

In [ ]:
# ── 2.6 Preprocessing leakage: selecting genes on all samples before cross-validation ──
rng = np.random.default_rng(SEED + 23)
X_noise, y_noise = rng.normal(size=(60, 3000)), rng.integers(0, 2, 60)  # 60 patients, labels are coin flips


def top_t_genes(X, y, k=20):
    """Indices of the k genes with the largest |t| between the two classes (uses the labels!)."""
    return np.argsort(-np.abs(stats.ttest_ind(X[y == 1], X[y == 0]).statistic))[:k]


folds = list(KFold(5, shuffle=True, random_state=SEED).split(X_noise))
genes_all = top_t_genes(X_noise, y_noise)  # ❌ selection sees every label
leaky = np.mean([LogisticRegression(max_iter=1000).fit(X_noise[tr][:, genes_all], y_noise[tr])
                 .score(X_noise[te][:, genes_all], y_noise[te]) for tr, te in folds])
honest = []
for tr, te in folds:  # ✅ selection repeated inside every fold
    g = top_t_genes(X_noise[tr], y_noise[tr])
    honest.append(LogisticRegression(max_iter=1000).fit(X_noise[tr][:, g], y_noise[tr]).score(X_noise[te][:, g], y_noise[te]))
X_new, y_new = rng.normal(size=(2000, 3000)), rng.integers(0, 2, 2000)  # fresh patients
truth_noise = LogisticRegression(max_iter=1000).fit(X_noise[:, genes_all], y_noise).score(X_new[:, genes_all], y_new)
print(f"CV accuracy, genes selected on ALL samples : {leaky:.3f}")
print(f"CV accuracy, selection inside each fold    : {np.mean(honest):.3f}")
print(f"accuracy on 2,000 fresh patients (truth)   : {truth_noise:.3f}   (labels are random: 0.5 is the best possible)")

🔍 **What just happened?** Selecting the 20 "best" genes on all 60 patients, then cross-validating, reports 0.90 accuracy **on
coin-flip labels**; selecting inside each fold gives 0.37, and 2,000 fresh patients give 0.50 — the truth. (0.37 is below 0.5
by chance: a CV estimate from 60 patients has a large standard error.) Among 3,000 noise genes some correlate with *these*
labels by chance; choosing them with the test patients' labels writes the answer into the features. The same leak comes from
oversampling a rare class *before* splitting, supervised batch correction, or tuning on the test set. Unsupervised steps
(scaling, PCA) leak far less, but the clean habit is cell 2.1's: a pipeline refitted in each fold.

**(d) Temporal leakage** has two forms. *Target leakage*: a feature recorded **after** the outcome ("antibiotics started" in
a sepsis model) is a near-perfect predictor in a retrospective table but unavailable at prediction time. *Drift*: when the
world changes (new assay vendor, new patient mix), a random split mixes past and future; a **forward-chaining** split (train
on the past, test on the following periods) mimics deployment. We simulate 10 quarters of admissions in which a lab test is
slowly recalibrated and the disease signature slowly rotates; development uses quarters 0–7, deployment 8–9.

In [ ]:
# ── 2.7 Temporal leakage: SIMULATED admissions over 10 quarters with drift ──
def simulate_admissions(rng, quarter, n=400, drift=0.2):
    """12 lab values; disease (25 %) shifts a direction that rotates with time; lab 0 drifts (recalibration).
    Also returns 'antibiotics started', which is recorded AFTER the diagnosis (target leakage)."""
    y = (rng.random(n) < 0.25).astype(int)
    w = np.zeros(12); w[0], w[1], w[2] = np.cos(drift * quarter), np.sin(drift * quarter), 0.6
    X = rng.normal(size=(n, 12)) + 1.2 * y[:, None] * w
    X[:, 0] += 0.15 * quarter
    antibiotics = np.where(y == 1, rng.random(n) < 0.85, rng.random(n) < 0.05).astype(float)
    return X, y, antibiotics, np.full(n, quarter)


rng = np.random.default_rng(SEED + 24)
adm = [simulate_admissions(rng, q) for q in range(10)]
X_adm, y_adm, abx, quarter = (np.concatenate([a[k] for a in adm]) for k in range(4))
dev_t, deploy_t = quarter < 8, quarter >= 8


def auroc_random_cv(X, y):
    """Mean AUROC over a random 5-fold split."""
    return np.mean([roc_auc_score(y[te], LogisticRegression(max_iter=1000).fit(X[tr], y[tr]).predict_proba(X[te])[:, 1])
                    for tr, te in KFold(5, shuffle=True, random_state=SEED).split(X)])


def auroc_forward(X, y, q, starts=range(3, 7)):
    """Forward chaining: train on quarters < t, test on quarters t and t + 1 (the deployment horizon)."""
    out = []
    for t in starts:
        m = LogisticRegression(max_iter=1000).fit(X[q < t], y[q < t])
        te = (q >= t) & (q <= t + 1)
        out.append(roc_auc_score(y[te], m.predict_proba(X[te])[:, 1]))
    return np.mean(out)


Xd, yd, qd = X_adm[dev_t], y_adm[dev_t], quarter[dev_t]
auc_target_leak = auroc_random_cv(np.c_[Xd, abx[dev_t]], yd)
auc_random_t = auroc_random_cv(Xd, yd)
auc_forward_t = auroc_forward(Xd, yd, qd)
deployed = LogisticRegression(max_iter=1000).fit(Xd, yd)
auc_truth_t = roc_auc_score(y_adm[deploy_t], deployed.predict_proba(X_adm[deploy_t])[:, 1])
print(f"AUROC, random CV with 'antibiotics started' : {auc_target_leak:.3f}   ← target leakage")
print(f"AUROC, random CV, labs only                 : {auc_random_t:.3f}")
print(f"AUROC, forward chaining, labs only          : {auc_forward_t:.3f}")
print(f"AUROC in deployment quarters 8–9 (truth)    : {auc_truth_t:.3f}")

In [ ]:
# ── 2.8 🧪 Summary: naive estimate vs honest estimate vs the truth on fresh data ──
summary = [("(a) donor leakage\naccuracy", acc_cells, acc_donors, acc_fresh, 0.5),
           ("(b) homology leakage\nR² (ridge)", homology.iloc[0, 1], homology.iloc[0, 2], homology.iloc[0, 3], 0.0),
           ("(c) preprocessing leakage\naccuracy", leaky, np.mean(honest), truth_noise, 0.5),
           ("(d) temporal leakage\nAUROC (labs only)", auc_random_t, auc_forward_t, auc_truth_t, 0.5)]
fig, axes = plt.subplots(1, 4, figsize=(12, 3.3))
for ax, (title, naive, fair, truth, floor) in zip(axes, summary):
    ax.bar([0, 1, 2], [naive, fair, truth], color=[OKABE_ITO[1], OKABE_ITO[0], OKABE_ITO[2]])
    for x, v in enumerate([naive, fair, truth]):
        ax.text(x, v + 0.01, f"{v:.2f}", ha="center", fontsize=8)
    ax.axhline(floor, color="0.5", ls=":", lw=1)
    ax.set_xticks([0, 1, 2]); ax.set_xticklabels(["naive", "honest", "truth"]); ax.set_title(title, fontsize=9)
    ax.set_ylim(min(floor, min(naive, fair, truth)) - 0.05, 1.0)
plt.suptitle("SIMULATED: leaky evaluations overstate performance; splits that match the claim track the truth", y=1.03)
plt.tight_layout(); plt.show()

🔍 **What just happened?** In every panel the naive estimate (orange) is the highest bar; the honest estimates (blue) are
much closer to the truth on fresh data (green), though noisy. Target leakage is the most extreme: "antibiotics started" lifts
the AUROC to 0.95, a number that would collapse at the bedside. For drift, forward chaining (0.78) is closer to the
deployment truth (0.72) than a random split (0.80) but still optimistic: the drift continues, and the deployed model, trained
on increasingly outdated data, lags further behind. Time-dependent data need **monitoring after deployment**.

✏️ **Your turn.** (1) In cell 2.3 set the clustering threshold to 0.35, then 0.95. What happens to the number of clusters,
the adjusted Rand index and the cluster-split R²? Why is the threshold part of the claim? (2) In cell 2.7 train the deployed
model on quarters 6–7 only. Is a smaller but more recent training set better here?

### 📄 Research connection — Pitfalls, leakage and reporting standards

- **What the field does.** Park & Marcotte (2012) showed that pair-input predictors (protein–protein, drug–target) are inflated when test
    pairs share components with training pairs; Whalen *et al.* (2022) catalogued five pitfalls in genomics (distributional
    differences, dependent examples, confounding, leaky preprocessing, unbalanced classes); Kapoor & Narayanan (2023)
    documented leakage across many fields; Geirhos *et al.* (2020) reviewed shortcut learning. Reporting standards such as
    DOME (Walsh *et al.* 2021) ask authors to state how data were split and how independent the splits are.
- **What this notebook builds.** Four leaks provoked on purpose and checked against fresh cohorts: donor-grouped CV, identity clustering of
    sequences (a miniature CD-HIT/MMseqs2) with `GroupKFold`, selection inside folds, forward chaining. Production
    equivalents: scikit-learn's group and time-series splitters, MMseqs2/CD-HIT clustering, scaffold and time splits for
    molecules ([Module 40](40_molecular_ml_drug_discovery.ipynb)), held-out chromosomes ([Module 36](36_regulatory_genomics_deep_learning.ipynb)).
- **Where the frontier is.** Foundation models add a new leak: a pre-trained model may already have seen your "test" sequences, cells or
    papers, and benchmark contamination is hard to audit ([Module 43](43_llms_ai_agents.ipynb)). How to split proteins (by
    identity, fold or function) and genomes (chromosomes share repeats and paralogues) remains debated; recent benchmarks ship
    fixed, documented splits so that numbers are comparable.
- **Read next.** Whalen *et al.* 2022, *Navigating the pitfalls of applying machine learning in genomics* (Nat Rev Genet);
    Kapoor & Narayanan 2023, *Leakage and the reproducibility crisis in ML-based science* (Patterns); Walsh *et al.* 2021,
    *DOME* (Nat Methods).

---
## 3 · Distribution shift: when the test data come from a different world

> 📍 **Where we are.** Leakage makes a test set *too similar* to the training set; distribution shift is the opposite
> problem in deployment — another batch, hospital or prevalence. We need words for kinds of shift, detection and corrections.

Write the joint distribution of inputs $x$ (a cell's profile) and labels $y$ (its type) as
$p(x,y)=p(x)\,p(y\mid x)=p(y)\,p(x\mid y)$. Three kinds of change are distinguished:

$$\text{covariate shift: } p_{\text{test}}(x)\neq p_{\text{train}}(x),\ p(y\mid x)\text{ fixed};\qquad
\text{label shift: } p_{\text{test}}(y)\neq p_{\text{train}}(y),\ p(x\mid y)\text{ fixed};\qquad
\text{concept drift: } p(y\mid x)\text{ changes}.$$

*In words:* under **covariate shift** inputs look different but the input→label rule is unchanged (a new batch shifts every
gene a little); under **label shift** classes change frequency but each looks the same (a tissue with more CD8 T cells);
under **concept drift** the rule itself changes (Section 2d). Real shifts are usually mixtures.

🔬 **Assay box — batches.** Cells processed on different days, chips, reagent lots or sites differ systematically (capture
efficiency, ambient RNA, depth); our simulator multiplies each gene by a batch-specific factor and changes depth.
[Module 31](31_batch_integration.ipynb) corrects such effects; here the classifier is *not* corrected.

Our cell-type classifier sees shallow data (median ≈ 130 UMIs per cell, so types overlap) from two strongly different
batches; it is trained on batch 1, and batch 2 plays the new site.

In [ ]:
# ── 3.1 SIMULATED cell-type data: two batches, shallow sequencing ──
rng = np.random.default_rng(SEED + 31)
pbmc = simulate_scrna(rng, n_cells=8000, n_genes=400, n_donors=4, n_batches=2, depth=120, batch_effect=1.2)
TYPES = list(PBMC_TYPES)
y_type = np.array([TYPES.index(t) for t in pbmc["obs"]["true_type"].to_numpy(str)])
batch = pbmc["obs"]["batch"].to_numpy(str)
X_pbmc = log_normalize(pbmc["counts"])
b1 = rng.permutation(np.flatnonzero(batch == "batch1"))
tr_c, cal_c, te_c = np.split(b1, [int(0.5 * len(b1)), int(0.7 * len(b1))])  # batch 1: train / calibration / test
b2 = np.flatnonzero(batch == "batch2")  # batch 2: the "new site"
centre = X_pbmc[tr_c].mean(0)
_, components, _ = pca(X_pbmc[tr_c], 50)  # PCA fitted on training cells only
Z_pbmc = (X_pbmc - centre) @ components.T
Z_pbmc /= Z_pbmc[tr_c].std(0)
print(f"SIMULATED: median {np.median(pbmc['obs']['n_counts']):.0f} UMIs per cell; "
      f"{len(tr_c)} training, {len(cal_c)} calibration, {len(te_c)} test cells (batch 1); {len(b2)} cells in batch 2")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
show = rng.choice(len(y_type), 3000, replace=False)
plot_embedding(Z_pbmc[show, :2], batch[show], ax=axes[0], title="PC 1–2 (fitted on batch 1), coloured by batch")
plot_embedding(Z_pbmc[show, :2], np.array(TYPES, dtype=object)[y_type[show]].astype(str), ax=axes[1],
               title="the same cells, coloured by true cell type")
plt.suptitle("SIMULATED PBMCs: batch 2 is displaced from batch 1, and T-cell subsets overlap", y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# ── 3.2 An over-parameterised classifier trained to 100 % training accuracy ──
class CellTypeMLP(nn.Module):
    """50 principal components → two hidden layers of 128 ReLU units → 6 cell-type logits."""

    def __init__(self, d_in=50, width=128, n_out=6):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, width), nn.ReLU(), nn.Linear(width, width), nn.ReLU(),
                                 nn.Linear(width, n_out))

    def forward(self, x):
        return self.net(x)


def train_mlp(Z, y, idx, epochs=60, seed=SEED):
    """Mini-batch Adam with no regularisation and no early stopping (on purpose: this over-fits)."""
    torch.manual_seed(seed)
    model, Zt, yt = CellTypeMLP(Z.shape[1]), torch.tensor(Z, dtype=torch.float32), torch.tensor(y)
    opt, idx = torch.optim.Adam(model.parameters(), lr=2e-3), torch.tensor(idx)
    for _ in range(epochs):
        perm = idx[torch.randperm(len(idx))]
        for k in range(0, len(perm), 128):
            b = perm[k:k + 128]
            loss = F.cross_entropy(model(Zt[b]), yt[b])
            opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        return model, model(Zt).double().numpy()  # also return the logits of every cell


t0 = time.time()
mlp, logits = train_mlp(Z_pbmc, y_type, tr_c)
for name, idx in [("training cells (batch 1)", tr_c), ("test cells (batch 1)", te_c), ("batch 2 (new site)", b2)]:
    print(f"accuracy on {name:25s}: {(logits[idx].argmax(1) == y_type[idx]).mean():.3f}")
print(f"(trained in {time.time() - t0:.1f} s)")

🔍 **What just happened?** The network memorises its training cells (accuracy 1.000), reaches 0.92 on new cells of the same
batch — the confusions are mostly between CD4 and CD8 T cells, whose profiles overlap at this depth — and drops to 0.68 in
batch 2: an uncorrected batch effect costs about 24 accuracy points. In practice you do not know the labels at the new site, so you would
not see this drop. Can we at least **detect** that the new data are different?

**A domain classifier** answers "could a model tell the two data sets apart?". Pool the reference inputs (label 0) and the new
inputs (label 1), train any classifier with cross-validation, and compute its AUROC. An AUROC near 0.5 means the classifier
cannot tell the sets apart; near 1 means they are clearly different. (Rabanser *et al.* 2019 compared many such two-sample
tests for shift detection.)

In [ ]:
# ── 3.3 Detecting covariate shift with a domain classifier ──
def domain_auroc(A, B, seed=SEED):
    """Cross-validated AUROC of a logistic regression separating rows of A (label 0) from rows of B (label 1)."""
    X = np.vstack([A, B]); d = np.r_[np.zeros(len(A)), np.ones(len(B))]
    p = cross_val_predict(LogisticRegression(max_iter=2000), X, d, cv=KFold(5, shuffle=True, random_state=seed),
                          method="predict_proba")[:, 1]
    return roc_auc_score(d, p)


b2_sample = rng.choice(b2, len(te_c), replace=False)
print(f"domain AUROC, batch-1 calibration vs batch-1 test cells: {domain_auroc(Z_pbmc[cal_c], Z_pbmc[te_c]):.3f}")
print(f"domain AUROC, batch-1 calibration vs batch-2 cells     : {domain_auroc(Z_pbmc[cal_c], Z_pbmc[b2_sample]):.3f}")

🔍 **What just happened?** Two random parts of batch 1 are indistinguishable (AUROC 0.48, chance level), while batch 2 is
separated perfectly (1.000). A domain classifier costs one line and should be run whenever a model meets new data.

**Label shift and its correction.** Suppose a new sample is a tissue rich in CD8 T cells and monocytes. Each cell type looks
as in training ($p(x\mid y)$ fixed) but the **proportions** differ. Two things go wrong: the naive estimate of the new
composition (count the predicted labels) is biased towards the training proportions, and the classifier's posteriors
$p_{\text{train}}(y\mid x)$ carry the wrong prior. Bayes' rule ([Module 02](02_probability_for_biologists.ipynb)) tells us how
to fix the posteriors if we knew the new prior $\pi_{\text{test}}$:

$$p_{\text{test}}(y\mid x)=\frac{p_{\text{train}}(y\mid x)\,\pi_{\text{test}}(y)/\pi_{\text{train}}(y)}
{\sum_{k}p_{\text{train}}(k\mid x)\,\pi_{\text{test}}(k)/\pi_{\text{train}}(k)}.$$

*In words:* multiply each class probability by how much more (or less) common that class has become, and renormalise. The
unknown $\pi_{\text{test}}$ is estimated by **EM** (Saerens, Latinne & Decaestecker 2002), exactly as the mixing weights of a
mixture model in [Module 13](13_mixture_models_em.ipynb): the **E-step** computes adjusted posteriors with the current
$\pi_{\text{test}}$; the **M-step** sets $\pi_{\text{test}}(k)$ to the average adjusted posterior of class $k$ over the new
cells; repeat until it stops changing. (The classifier should be calibrated for this to work well — we use the
temperature-scaled probabilities of Section 4, computed in the next cell ahead of time.)

In [ ]:
# ── 3.4 Label shift: estimate the new composition by EM (Saerens et al. 2002) ──
def softmax_T(logits, T=1.0):
    """Class probabilities from logits at temperature T (Section 4)."""
    z = logits / T
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)


def em_label_shift(P, pi_train, n_iter=200):
    """Estimate test-set class proportions from calibrated training posteriors P (n × K) by EM."""
    pi = pi_train.copy()
    for _ in range(n_iter):
        W = P * pi / pi_train
        W /= W.sum(1, keepdims=True)  # E-step: posteriors under the current prior
        pi = W.mean(0)  # M-step: new prior = mean posterior
    return pi, W


T_fit = minimize_scalar(lambda T: -np.log(softmax_T(logits[cal_c], T)[np.arange(len(cal_c)), y_type[cal_c]]).mean(),
                        bounds=(0.05, 20), method="bounded").x  # temperature (explained in Section 4)
target = np.array([0.05, 0.40, 0.05, 0.10, 0.30, 0.10])  # a CD8- and monocyte-rich tissue
shifted = np.concatenate([rng.choice(te_c[y_type[te_c] == k], int(450 * target[k]), replace=False) for k in range(6)])
P_shift = softmax_T(logits[shifted], T_fit)
pi_train = np.bincount(y_type[tr_c], minlength=6) / len(tr_c)
pi_true = np.bincount(y_type[shifted], minlength=6) / len(shifted)
pi_naive = np.bincount(P_shift.argmax(1), minlength=6) / len(shifted)
pi_em, P_adj = em_label_shift(P_shift, pi_train)
comp = pd.DataFrame({"training": pi_train, "true (new tissue)": pi_true, "naive count": pi_naive, "EM estimate": pi_em},
                    index=TYPES)
print(comp.round(3).T.to_string())
print(f"\nmean absolute error of the composition: naive {np.abs(pi_naive - pi_true).mean():.3f}, "
      f"EM {np.abs(pi_em - pi_true).mean():.3f}")
print(f"accuracy on the new tissue: before {(P_shift.argmax(1) == y_type[shifted]).mean():.3f}, "
      f"after prior correction {(P_adj.argmax(1) == y_type[shifted]).mean():.3f}")
print(f"domain AUROC (PCs) for this label shift: {domain_auroc(Z_pbmc[cal_c], Z_pbmc[shifted]):.3f}; "
      f"on predicted probabilities: {domain_auroc(softmax_T(logits[cal_c], T_fit), P_shift):.3f}")

In [ ]:
# ── 3.5 The composition of the new tissue: truth vs naive vs EM ──
fig, ax = plt.subplots(figsize=(8, 3.2))
w = 0.2
for k, (col, c) in enumerate(zip(["training", "true (new tissue)", "naive count", "EM estimate"],
                                  ["0.7", OKABE_ITO[2], OKABE_ITO[1], OKABE_ITO[0]])):
    ax.bar(np.arange(6) + (k - 1.5) * w, comp[col], w, color=c, label=col)
ax.set_xticks(range(6)); ax.set_xticklabels(TYPES); ax.set_ylabel("fraction of cells")
ax.set_title("SIMULATED label shift: counting predictions is biased towards the training mix; EM corrects most of it")
ax.legend(ncol=4, fontsize=7); plt.tight_layout(); plt.show()

🔍 **What just happened?** The new tissue has 40 % CD8 and 5 % CD4 T cells. Counting predicted labels says 35 % and 10 %:
CD4/CD8 confusions push the estimate back towards the balanced training mix. EM recovers most of the difference
(38 % and 8 %), the mean composition error halves (0.019 → 0.010), and re-weighting the posteriors with the estimated prior
raises accuracy on the new tissue from 0.91 to 0.94 — for free, without a single new label. Note the last line: a domain
classifier on the 50 PCs notices this shift only weakly (AUROC 0.58), one on the predicted probabilities somewhat better
(0.65), compared with 1.000 for the batch. **A domain AUROC close to 0.5 does not prove that nothing changed**: label
shift moves cells between existing, well-separated classes without creating new kinds of cells. Comparing the distribution
of *predicted labels* with the training one is a more sensitive check.

⚠️ **Assumptions matter.** The EM correction assumes each cell type looks the same in the new data. If disease also changes
what monocytes look like (concept and covariate shift at once — the stimulated monocytes of Module 30), the correction can
make things worse.

✏️ **Your turn.** Implement the confusion-matrix method of Lipton, Wang & Smola (2018) ("black-box shift estimation"):
estimate the confusion matrix $C_{ij}=P(\hat y=i, y=j)$ on the calibration cells, the distribution $\mu_i=P(\hat y=i)$ of
predictions on the new tissue, and solve $C\,w=\mu$ for the weights $w_j=\pi_{\text{test}}(j)/\pi_{\text{train}}(j)$. Compare its
composition estimate with EM.

---
## 4 · Calibration: can we believe the probabilities?

> 📍 **Where we are.** A classifier outputs probabilities, and decisions use them — "send for confirmatory testing if
> P(disease) > 0.2", "call this cell a CD8 T cell only if P > 0.9". That only makes sense if the probabilities are
> **calibrated**: among all predictions made with confidence 0.8, about 80 % should be correct.

**Reliability diagram and ECE.** Sort predictions into bins by their confidence (the largest class probability), and in each
bin compare the average confidence with the accuracy. A calibrated model lies on the diagonal. The **expected calibration
error** summarises the gaps:

$$\mathrm{ECE}=\sum_{b=1}^{B}\frac{n_b}{n}\,\big|\operatorname{acc}(b)-\operatorname{conf}(b)\big|.$$

*In words:* for each confidence bin $b$ holding $n_b$ of the $n$ predictions, take the absolute difference between how often
the model was right and how sure it claimed to be, and average these gaps weighted by bin size. ECE = 0 means perfect
calibration; an over-confident model has accuracy below confidence.

**Temperature scaling** (Guo *et al.* 2017) divides all logits by one number $T>0$ before the softmax,
$p_k=e^{z_k/T}/\sum_j e^{z_j/T}$, choosing $T$ to minimise the negative log-likelihood on a held-out **calibration set**.
*In words:* $T>1$ softens over-confident probabilities, $T<1$ sharpens under-confident ones; because dividing by $T$ does not
change which logit is largest, accuracy is unchanged — only the confidence moves. It is the simplest member of a family
that includes **Platt scaling** (below).

In [ ]:
# ── 4.1 Expected calibration error and reliability curves ──
def ece_score(P, y, n_bins=10):
    """Expected calibration error of class probabilities P (n × K) against labels y."""
    conf, correct = P.max(1), P.argmax(1) == y
    edges, total = np.linspace(0, 1, n_bins + 1), 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        in_bin = (conf > lo) & (conf <= hi)
        if in_bin.any():  # weight |accuracy − confidence| by the bin's share
            total += in_bin.mean() * abs(correct[in_bin].mean() - conf[in_bin].mean())
    return total


def reliability_curve(P, y, n_bins=10):
    """Mean confidence, accuracy and count in each confidence bin (for plotting)."""
    conf, correct = P.max(1), P.argmax(1) == y
    bins = np.minimum((conf * n_bins).astype(int), n_bins - 1)
    return np.array([(conf[bins == b].mean(), correct[bins == b].mean(), (bins == b).sum())
                     for b in range(n_bins) if (bins == b).sum() >= 10])  # skip nearly empty bins (too noisy)


print(f"fitted temperature on the calibration cells: T = {T_fit:.2f}")
rows = []
for name, idx in [("test (batch 1)", te_c), ("batch 2 (shift)", b2)]:
    P1, PT = softmax_T(logits[idx]), softmax_T(logits[idx], T_fit)
    rows.append((name, (P1.argmax(1) == y_type[idx]).mean(), P1.max(1).mean(), ece_score(P1, y_type[idx]),
                 PT.max(1).mean(), ece_score(PT, y_type[idx])))
calib = pd.DataFrame(rows, columns=["cells", "accuracy", "mean confidence", "ECE", "mean confidence (T)", "ECE (T)"])
print(calib.round(3).to_string(index=False))

In [ ]:
# ── 4.2 Reliability diagrams before and after temperature scaling, and under shift ──
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
panels = [(softmax_T(logits[te_c]), y_type[te_c], "batch-1 test, raw softmax"),
          (softmax_T(logits[te_c], T_fit), y_type[te_c], f"batch-1 test, temperature T = {T_fit:.1f}"),
          (softmax_T(logits[b2], T_fit), y_type[b2], f"batch 2 (shift), same T = {T_fit:.1f}")]
for ax, (P, y_, title) in zip(axes, panels):
    curve = reliability_curve(P, y_)
    ax.plot([0, 1], [0, 1], "k--", lw=0.8)
    ax.bar(curve[:, 0], curve[:, 1], width=0.08, color=OKABE_ITO[0], alpha=0.75, label="accuracy in bin")
    ax.scatter(curve[:, 0], curve[:, 0], marker="_", s=200, color=OKABE_ITO[1], label="perfect calibration")
    ax.set_xlim(0, 1.02); ax.set_ylim(0, 1.02); ax.set_xlabel("confidence (largest probability)"); ax.set_ylabel("accuracy")
    ax.set_title(f"{title}\nECE = {ece_score(P, y_):.3f}")
axes[0].legend(loc="upper left", fontsize=7)
plt.suptitle("SIMULATED cell-type classifier: over-confident, fixed by one temperature, worse again under shift "
             "(bins with ≥ 10 cells)", y=1.03)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The raw network is **over-confident**: on batch-1 test cells it is right 92 % of the time but
claims an average confidence of 0.97 (ECE 0.058). Over-confidence is typical of networks trained to zero training loss,
which push their logits ever further apart (Guo *et al.* 2017). A single temperature, T = 2.13, fitted on the calibration
cells, brings the average confidence down to 0.93 and the ECE to 0.018 without changing a single prediction. Under shift
(batch 2) the same temperature helps only a little — ECE falls from 0.25 to 0.17 — and calibration is far worse than in the
batch it was fitted on: the model is right only 68 % of the time while claiming 85 %. Calibration is a property of a model **and a
data distribution** (Ovadia *et al.* 2019 found the same for many methods and shifts).

**Platt scaling** (Platt 1999) was invented for support vector machines ([Module 11](11_support_vector_machines.ipynb)),
whose decision values $s(x)$ are distances to the margin, not probabilities. It fits

$$P(y=1\mid s)=\sigma(a\,s+b)=\frac{1}{1+e^{-(a s+b)}}$$

*In words:* a one-feature logistic regression ([Module 09](09_logistic_regression.ipynb)) on the calibration set maps the
SVM's score to a probability; $a$ sets how quickly confidence grows with the distance from the boundary, $b$ shifts the
boundary. (Temperature scaling is the multi-class analogue with $b=0$ and one shared slope $a=1/T$.)

In [ ]:
# ── 4.3 Platt scaling of a linear SVM: CD4 vs CD8 T cells ──
t_cells = lambda idx: idx[np.isin(y_type[idx], [0, 1])]  # CD4 T (0) and CD8 T (1) only
tr_t, cal_t, te_t = t_cells(tr_c), t_cells(cal_c), t_cells(te_c)
svm = LinearSVC(C=0.1, max_iter=5000).fit(Z_pbmc[tr_t], y_type[tr_t])
s_cal, s_te = svm.decision_function(Z_pbmc[cal_t]), svm.decision_function(Z_pbmc[te_t])
platt = LogisticRegression().fit(s_cal[:, None], y_type[cal_t])  # σ(a·s + b) fitted on calibration cells
p_naive = sigmoid(s_te)  # "just squash the score" — not a probability
p_platt = platt.predict_proba(s_te[:, None])[:, 1]
two_col = lambda p: np.c_[1 - p, p]
print(f"SVM accuracy on test T cells: {(svm.predict(Z_pbmc[te_t]) == y_type[te_t]).mean():.3f}")
print(f"Platt parameters: a = {platt.coef_[0, 0]:.2f}, b = {platt.intercept_[0]:.2f}")
print(f"ECE  sigmoid(score): {ece_score(two_col(p_naive), y_type[te_t]):.3f}   Platt: {ece_score(two_col(p_platt), y_type[te_t]):.3f}")

🔍 **What just happened?** Squashing the raw SVM score through a sigmoid gives numbers between 0 and 1, but they are not
probabilities: the ECE is 0.12. Platt's two parameters, fitted on calibration cells, reduce it to 0.018. The fitted slope
$a = 2.35$ says that this SVM's scores are too timid — one unit of score corresponds to about 2.35 units of log-odds — and
$b\approx0$ says its boundary was already in the right place.

⚠️ **Pitfalls.** (1) Calibrate on data the model was **not** trained on — training-set probabilities of an over-fitted model
are all near 1 (Section 3.2). (2) A low ECE on average can hide large miscalibration in a subgroup (a rare cell type, a
minority population; Section 8). (3) ECE depends on the number of bins and is noisy for small test sets; report it with the
reliability diagram.

✏️ **Your turn.** Label 200 random batch-2 cells (pretend an expert annotated them), refit the temperature on those cells only
and recompute the batch-2 ECE. How much of the calibration loss under shift does a small labelled sample from the new site
recover?

---
## 5 · Uncertainty: ensembles, aleatoric and epistemic

> 📍 **Where we are.** Calibration asks whether stated confidence is right *on average*. Now we ask *where* a model is
> uncertain and *why*. We switch to regression — predicting the potency of molecules (Module 40) — because there the two
> sources of uncertainty can be separated cleanly and, in a simulation, checked against the truth.

**Two kinds of uncertainty.** *Aleatoric* uncertainty is noise in the measurement itself: replicate potency assays disagree,
and no amount of data removes that spread. *Epistemic* uncertainty is the model's ignorance: with few or no similar
training examples, many different models fit the data, and they disagree about the new case; more data reduce it. A **deep
ensemble** (Lakshminarayanan, Pritzel & Blundell 2017) trains $M$ networks from different random initialisations, each
predicting a mean $\mu_m(x)$ and a variance $\sigma_m^2(x)$ by minimising the Gaussian negative log-likelihood
$\tfrac12\log\sigma_m^2(x)+\tfrac{(y-\mu_m(x))^2}{2\sigma_m^2(x)}$.

🧮 **Math refresher — the law of total variance.** For a prediction that averages over models $m$,

$$\operatorname{Var}[y\mid x]=\underbrace{\tfrac1M\textstyle\sum_m\sigma_m^2(x)}_{\text{aleatoric}}+
\underbrace{\tfrac1M\textstyle\sum_m\big(\mu_m(x)-\bar\mu(x)\big)^2}_{\text{epistemic}},\qquad
\bar\mu(x)=\tfrac1M\textstyle\sum_m\mu_m(x).$$

*In words:* the total predictive variance is the average noise each model expects (aleatoric) plus the spread of the models'
means around their average (epistemic). It is the decomposition $\operatorname{Var}[Y]=E[\operatorname{Var}(Y\mid M)]+
\operatorname{Var}(E[Y\mid M])$ from probability, with "which model" as the random variable.

> 🔗 **Builds on** [Module 21 · Bayesian learning](21_bayesian_learning_mcmc.ipynb): the Bayesian posterior predictive
> averages the predictions of all parameter values weighted by the posterior; an ensemble is a cheap approximation with
> $M$ "samples" obtained by optimisation from random starts instead of MCMC.

🔬 **Assay box — potency noise.** pIC50 values come from dose–response curves; replicate measurements typically scatter by a
few tenths of a log unit, more for compounds that are poorly soluble or aggregate in the assay buffer. We use the oracle
potency of `simulate_molecules` and add our own noise whose standard deviation **grows with lipophilicity** (logP), from
0.15 for polar molecules to about 0.5 at logP = 3 — so the true aleatoric uncertainty of every molecule is known.

We represent each molecule by its scaffold and the substituent at each site (the classic **Free–Wilson** encoding,
Free & Wilson 1964) plus six descriptors. Three test sets: new molecules from the same chemistry ("in distribution"),
molecules from **the rarest scaffolds, never seen** in training, and molecules **more lipophilic** than any training molecule
(logP > 3).

In [ ]:
# ── 5.1 SIMULATED potency data with known, heteroscedastic noise ──
rng = np.random.default_rng(SEED + 51)
mol = simulate_molecules(rng, n_molecules=2500, noise=0.0)  # noise=0 → pIC50 is the oracle value
tab = mol["table"]
logp = tab["logp"].to_numpy()
noise_sd = 0.15 + 0.125 * np.clip(logp, 0, None)  # TRUE aleatoric sd of every molecule
y_true_mol = tab["pIC50"].to_numpy()
y_mol = y_true_mol + rng.normal(0, noise_sd)  # what the assay reports


def free_wilson_features(table):
    """One-hot scaffold + one-hot (site, substituent) + six standardised descriptors."""
    scaffolds, subs = sorted(table["scaffold"].unique()), list(MOLECULE_SUBSTITUENTS)
    fw = np.zeros((len(table), len(scaffolds) + 4 * len(subs)))
    for i, (sc, st) in enumerate(zip(table["scaffold"].to_numpy(str), table["substituents"].to_numpy(str))):
        fw[i, scaffolds.index(sc)] = 1
        for part in st.split("; "):
            site, name = part.split(":")
            fw[i, len(scaffolds) + int(site[4:]) * len(subs) + subs.index(name)] = 1
    desc = table[["mol_weight", "logp", "hbd", "hba", "heavy_atoms", "aromatic_atoms"]].to_numpy(float)
    return np.hstack([fw, (desc - desc.mean(0)) / desc.std(0)]).astype(np.float32)


X_mol = free_wilson_features(tab)
scaf = tab["scaffold"].to_numpy(str)
counts = tab["scaffold"].value_counts(ascending=True)
held_out = list(counts.index[np.cumsum(counts.to_numpy()) <= 300])  # the rarest scaffolds (≈ 10 % of molecules)
new_scaffold = np.isin(scaf, held_out)
high_logp = logp > 3.0
seen = rng.permutation(np.flatnonzero(~new_scaffold & ~high_logp))
tr_m, pool_m = seen[:1100], seen[1100:]  # training / in-distribution pool
te_scaf, te_logp = np.flatnonzero(new_scaffold & ~high_logp), np.flatnonzero(high_logp & ~new_scaffold)
print(f"SIMULATED: {X_mol.shape[1]} features; {len(tr_m)} training molecules, {len(pool_m)} in-distribution, "
      f"{len(te_scaf)} with new scaffolds {held_out}, {len(te_logp)} with logP > 3")
print(f"true noise sd: {noise_sd[tr_m].min():.2f}–{noise_sd[tr_m].max():.2f} in training, "
      f"mean {noise_sd[te_logp].mean():.2f} for the logP > 3 molecules")

In [ ]:
# ── 5.2 A deep ensemble of mean–variance networks ──
class MeanVarianceNet(nn.Module):
    """Predicts a mean and a log-variance for every input."""

    def __init__(self, d_in, width=16):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(d_in, width), nn.ReLU(), nn.Linear(width, width), nn.ReLU(), nn.Linear(width, 2))

    def forward(self, x):
        out = self.body(x)
        return out[:, 0], out[:, 1].clamp(-8, 4)


def train_mean_variance(X, y, idx, seed, steps=300, lr=5e-3, wd=1e-2):
    """Full-batch Adam on the Gaussian negative log-likelihood; returns mean and variance for every row of X."""
    torch.manual_seed(seed)
    net, Xt = MeanVarianceNet(X.shape[1]), torch.tensor(X)
    y_mu, y_sd = y[idx].mean(), y[idx].std()
    yt = torch.tensor((y[idx] - y_mu) / y_sd, dtype=torch.float32)
    opt = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=wd)
    for _ in range(steps):
        m, log_var = net(Xt[idx])
        loss = (0.5 * log_var + 0.5 * (yt - m) ** 2 / log_var.exp()).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        m, log_var = net(Xt)
    return m.numpy() * y_sd + y_mu, np.exp(log_var.numpy()) * y_sd ** 2


t0 = time.time()
members = [train_mean_variance(X_mol, y_mol, tr_m, seed=SEED + m) for m in range(5)]
MU = np.array([m for m, _ in members]); VAR = np.array([v for _, v in members])
mu_ens, aleatoric, epistemic = MU.mean(0), VAR.mean(0), MU.var(0)  # the law of total variance
print(f"5 ensemble members trained in {time.time() - t0:.1f} s")

In [ ]:
# ── 5.3 🧪 Ground truth: error, aleatoric and epistemic uncertainty, and interval coverage ──
z90 = stats.norm.ppf(0.95)  # ±1.645 sd = central 90 % of a Gaussian
rows = []
for name, idx in [("in distribution", pool_m), ("new scaffolds", te_scaf), ("logP > 3 (extrapolation)", te_logp)]:
    sd_total = np.sqrt(aleatoric[idx] + epistemic[idx])
    rows.append((name, np.sqrt(((mu_ens[idx] - y_true_mol[idx]) ** 2).mean()), noise_sd[idx].mean(),
                 np.sqrt(aleatoric[idx]).mean(), np.sqrt(epistemic[idx]).mean(),
                 (np.abs(MU[0, idx] - y_mol[idx]) < z90 * np.sqrt(VAR[0, idx])).mean(),
                 (np.abs(mu_ens[idx] - y_mol[idx]) < z90 * sd_total).mean()))
unc = pd.DataFrame(rows, columns=["test set", "RMSE vs truth", "true noise sd", "aleatoric sd", "epistemic sd",
                                  "90% coverage: one net", "90% coverage: ensemble"])
print(unc.round(3).to_string(index=False))
r_ale = np.corrcoef(np.sqrt(aleatoric[pool_m]), noise_sd[pool_m])[0, 1]
print(f"\ncorrelation of estimated aleatoric sd with the TRUE noise sd (in distribution): {r_ale:.2f}")

In [ ]:
# ── 5.4 What the ensemble knows and what it does not ──
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
a1.scatter(noise_sd[pool_m], np.sqrt(aleatoric[pool_m]), s=8, alpha=0.5, color=OKABE_ITO[0])
lim = [0, max(noise_sd[pool_m].max(), np.sqrt(aleatoric[pool_m]).max()) * 1.05]
a1.plot(lim, lim, "k--", lw=0.8); a1.set_xlim(lim); a1.set_ylim(lim)
a1.set_xlabel("TRUE noise sd of the molecule"); a1.set_ylabel("estimated aleatoric sd")
a1.set_title(f"aleatoric part tracks the true noise (r = {r_ale:.2f})\nbut is too small on average")
x = np.arange(3); w = 0.2
bars = [("true noise sd", "true noise sd", "#9ecae1"), ("aleatoric sd", "estimated aleatoric sd", OKABE_ITO[0]),
        ("RMSE vs truth", "error of the mean (RMSE vs truth)", "#fdae6b"), ("epistemic sd", "estimated epistemic sd", OKABE_ITO[1])]
for k, (col, label, c) in enumerate(bars):
    a2.bar(x + (k - 1.5) * w, unc[col], w, color=c, label=label)
a2.set_xticks(x); a2.set_xticklabels(["in distribution", "new scaffolds", "logP > 3"]); a2.set_ylabel("pIC50 units")
a2.set_title("blue pair should match (noise); orange pair should match (ignorance)"); a2.legend(fontsize=7)
plt.suptitle("SIMULATED potency assay: a deep ensemble of 5 mean–variance networks", y=1.02)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Read the table row by row.

Two comparisons matter: the **aleatoric** sd should match the true noise sd (blue pair in the figure), and the
**epistemic** sd should match the error of the ensemble mean against the noise-free truth (orange pair).

* **In distribution** the ensemble mean is accurate (RMSE 0.11 against the truth) and the epistemic sd (0.09) is of the
  right size. The aleatoric estimate tracks the true per-molecule noise well (r = 0.88) — the networks learned that
  lipophilic molecules are measured less precisely — but it is **too small** on average (0.22 vs a true 0.27), so nominal
  90 % intervals cover 84 % of measurements. Mean–variance networks trained by likelihood tend to under-estimate variance on
  the points they fit; a single network is worse (74 %). Ensembling helps, but does not calibrate.
* **New scaffolds:** the error explodes (RMSE 1.02) because the scaffold's baseline potency is simply unknown — yet the
  epistemic sd barely moves (0.10 vs 0.09), ten times smaller than the error. All five networks see an all-zero input for
  "a scaffold I never saw", and weight decay drives them to nearly the *same* answer for it. Coverage collapses to 12 %.
* **logP > 3:** extrapolating a continuous descriptor is where networks disagree — the epistemic sd doubles (0.18) — but the
  error is 0.72; and the aleatoric estimate (0.48) under-estimates the true noise of these greasy molecules (0.69), which it
  can only extrapolate. Coverage: 65 %.

So uncertainty estimates help rank predictions within the training distribution, but "the model is unsure" fails as a
safety alarm exactly where it is most needed. Ovadia *et al.* (2019) reached the same conclusion across many methods and
shifts (ensembles were among the best, not good enough). This motivates two complementary tools: an applicability-domain
check (distance to training data, Module 40) and **conformal prediction**, which corrects the coverage of any model.

⚠️ **Pitfall.** Never read a single network's softmax or variance output as "the uncertainty". It omits epistemic
uncertainty entirely, and for a network trained by likelihood even its aleatoric part is usually too small.

✏️ **Your turn.** Retrain the ensemble with `wd=0` (no weight decay) in `train_mean_variance`. Does the epistemic sd on the
new scaffolds increase? Why would random initial weights for never-active inputs make the members disagree more?

### 📄 Research connection — Calibration and uncertainty quantification

- **What the field does.** Platt (1999) turned SVM scores into probabilities; Guo *et al.* (2017) showed that modern deep networks are
    over-confident and that temperature scaling fixes much of it. Bayesian deep learning (MC dropout, Gal & Ghahramani 2016;
    variational and Laplace approximations) and deep ensembles (Lakshminarayanan *et al.* 2017) estimate epistemic
    uncertainty; Kendall & Gal (2017) popularised the aleatoric/epistemic split. Ovadia *et al.* (2019) benchmarked these
    methods under dataset shift and found that calibration degrades for all of them, ensembles least.
- **What this notebook builds.** ECE and reliability diagrams from scratch, temperature scaling by one-dimensional likelihood optimisation, Platt
    scaling with a logistic regression, and a 5-member deep ensemble of mean–variance networks whose aleatoric estimate is
    checked against the **known** simulated noise. Production tools: `CalibratedClassifierCV` in scikit-learn, uncertainty
    heads and ensembles in PyTorch; in molecular property prediction, Chemprop offers ensembles and evidential uncertainty.
- **Where the frontier is.** Uncertainty that is reliable under shift remains open: distance-aware networks, evidential regression and
    Bayesian last layers are active areas, and in biology uncertainty is increasingly used to *act* — choosing which molecules
    or variants to test next (Bayesian optimisation and active learning, [Module 29](29_gaussian_processes_bayesian_optimization.ipynb)).
    Calibration of foundation-model outputs and of large language models ([Module 43](43_llms_ai_agents.ipynb)) is a newer
    and largely unsolved problem.
- **Read next.** Guo *et al.* 2017, *On calibration of modern neural networks* (ICML); Lakshminarayanan, Pritzel & Blundell 2017,
    *Deep ensembles* (NeurIPS); Ovadia *et al.* 2019, *Can you trust your model's uncertainty?* (NeurIPS).

---
## 6 · 🧪 Conformal prediction: a coverage guarantee for any model

> 📍 **Where we are.** Section 5 ended with intervals that claimed 90 % but covered 84 % — and 12 % for new scaffolds.
> Conformal prediction fixes this with almost no assumptions: it uses a held-out calibration set to measure how wrong the
> model actually is, and widens (or narrows) its predictions just enough. It works for any model — ridge, random forest, transformer — and any data,
> as long as the calibration and test points are **exchangeable** (e.g. drawn independently from the same distribution).

**Split conformal regression** (Vovk, Gammerman & Shafer 2005; Lei *et al.* 2018). Train a model $\hat\mu$ on the training
set. On $n$ calibration points compute **nonconformity scores** $s_i=|y_i-\hat\mu(x_i)|$. Let $\hat q$ be the
$\lceil (n+1)(1-\alpha)\rceil$-th smallest score. For a new $x$ predict the interval

$$\hat C(x)=\big[\hat\mu(x)-\hat q,\ \hat\mu(x)+\hat q\big],\qquad
P\big(Y_{\text{new}}\in\hat C(X_{\text{new}})\big)\ \ge\ 1-\alpha.$$

*In words:* find the residual size that $(1-\alpha)$ of calibration points stay within, with a tiny upward correction, and
put a band of that half-width around every prediction; then a new point falls inside its band with probability at least
$1-\alpha$ — whatever the model and whatever the distribution of the data.

🧮 **Why the guarantee holds.** If the $n$ calibration scores and the new point's score are exchangeable, the new score is
equally likely to have any rank among the $n+1$ scores. It exceeds $\hat q$ only if its rank is among the top
$n+1-\lceil(n+1)(1-\alpha)\rceil\le\alpha(n+1)$, which happens with probability at most $\alpha$. That is the whole proof —
no Gaussian assumption, no asymptotics. Two caveats are built in: the probability is over the random calibration set *and*
the test point (any particular calibration set can be a little lucky or unlucky), and it is **marginal** — an average over
all test points, not a promise for each subgroup.

In [ ]:
# ── 6.1 Split conformal intervals and their coverage over 500 random calibration/test splits ──
def conformal_quantile(scores, alpha):
    """The ceil((n + 1)(1 − α))-th smallest calibration score (∞ if that rank exceeds n)."""
    n = len(scores)
    k = int(np.ceil((n + 1) * (1 - alpha)))
    return np.inf if k > n else np.sort(scores)[k - 1]


ALPHA, N_CAL = 0.10, 300
resid = np.abs(y_mol - mu_ens)  # scores of the ensemble mean
rng = np.random.default_rng(SEED + 61)
coverage, halfwidth = [], []
for _ in range(500):
    perm = rng.permutation(pool_m)
    cal, test = perm[:N_CAL], perm[N_CAL:]
    q = conformal_quantile(resid[cal], ALPHA)
    coverage.append((resid[test] <= q).mean()); halfwidth.append(q)
coverage = np.array(coverage)
l = int(np.floor((N_CAL + 1) * ALPHA))
theory = stats.beta(N_CAL + 1 - l, l)  # distribution of coverage (infinite test set)
print(f"nominal coverage {1 - ALPHA:.2f}; conformal coverage over 500 splits: mean {coverage.mean():.3f}, "
      f"5th–95th percentile {np.percentile(coverage, 5):.3f}–{np.percentile(coverage, 95):.3f}")
print(f"theory (Beta({N_CAL + 1 - l}, {l})): mean {theory.mean():.3f}, 5th–95th percentile "
      f"{theory.ppf(0.05):.3f}–{theory.ppf(0.95):.3f}")
print(f"median interval: prediction ± {np.median(halfwidth):.2f} pIC50 units")

🧮 **Math refresher — the distribution of coverage.** For a fixed calibration set of size $n$ the coverage on an infinitely
large test set is itself random; with continuous scores it follows a $\mathrm{Beta}(n+1-l,\ l)$ distribution with
$l=\lfloor (n+1)\alpha\rfloor$ (Vovk 2012; Angelopoulos & Bates 2023). *In words:* on average the coverage is
$(n+1-l)/(n+1)\ge1-\alpha$, and it fluctuates around that value by roughly $\sqrt{\alpha(1-\alpha)/n}$ — about ±2 % for
$n=300$ — so larger calibration sets give coverage closer to the nominal level on every run. Our test sets are finite, which
adds a little extra spread.

In [ ]:
# ── 6.2 🧪 Coverage over 500 splits vs theory; conditional coverage by noise level ──
noise_group = np.digitize(noise_sd, [0.151, 0.34])  # true noise: 0 = 0.15 (polar), 1 = 0.15–0.34, 2 = > 0.34 (lipophilic)
sd_ens = np.sqrt(aleatoric + epistemic)
norm_scores = resid / sd_ens  # normalised (adaptive) scores
perm = np.random.default_rng(SEED + 62).permutation(pool_m)
cal, test = perm[:N_CAL], perm[N_CAL:]
q_abs, q_norm = conformal_quantile(resid[cal], ALPHA), conformal_quantile(norm_scores[cal], ALPHA)
cond = pd.DataFrame([(t, (resid[test][noise_group[test] == t] <= q_abs).mean(),
                      (norm_scores[test][noise_group[test] == t] <= q_norm).mean(),
                      2 * q_abs, 2 * (q_norm * sd_ens[test][noise_group[test] == t]).mean()) for t in range(3)],
                    columns=["noise group", "coverage |y−μ|", "coverage |y−μ|/σ", "width |y−μ|", "mean width |y−μ|/σ"])
cond["test molecules"] = [int((noise_group[test] == t).sum()) for t in range(3)]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.4))
a1.hist(coverage, bins=30, density=True, color=OKABE_ITO[0], alpha=0.7, label="500 random splits")
xs = np.linspace(0.8, 1.0, 300)
a1.plot(xs, theory.pdf(xs), color=OKABE_ITO[1], lw=2, label="Beta theory")
a1.axvline(1 - ALPHA, color="k", ls="--", lw=1, label="nominal 0.90")
a1.set_xlabel("coverage of the test molecules"); a1.set_ylabel("density"); a1.legend(fontsize=7)
a1.set_title("coverage over 500 splits is centred on the nominal 0.90")
x = np.arange(3)
a2.plot(x, cond.iloc[:, 1], "o-", ms=8, color=OKABE_ITO[3], label="score |y − μ| (constant width)")
a2.plot(x, cond.iloc[:, 2], "s-", ms=8, color=OKABE_ITO[0], label="score |y − μ| / σ (adaptive width)")
a2.axhline(1 - ALPHA, color="k", ls="--", lw=1)
a2.set_xticks(x); a2.set_xticklabels(["sd 0.15 (polar)", "sd 0.15–0.34", "sd > 0.34 (lipophilic)"]); a2.set_ylim(0.5, 1.02)
a2.set_xlim(-0.4, 2.4); a2.set_ylabel("coverage"); a2.set_title("marginal ≠ conditional coverage")
a2.legend(fontsize=7, loc="lower left")
plt.suptitle("SIMULATED potency assay: split conformal prediction around the ensemble mean", y=1.02)
plt.tight_layout(); plt.show()
print(cond.round(3).to_string(index=False))

🔍 **What just happened?** 🧪 Over 500 random calibration/test splits the coverage averaged 0.900 and stayed between 0.85 and
0.94 (5th–95th percentile) — close to the Beta curve predicted by theory (0.87–0.93), slightly wider because each test set is
finite (309 molecules) — even though the same model's Gaussian intervals covered only 84 %. Conformal prediction did not
make the model better; it **measured** how wrong the model is and sized the intervals accordingly (median: prediction ±
0.47 pIC50 units).

The right panel shows the limit of a *marginal* guarantee. With the plain score $|y-\hat\mu|$ every interval has the same
width, so the least noisy molecules are over-covered (0.99) and the noisiest under-covered (0.75) — the average is 0.90,
but a chemist working on lipophilic compounds would be misled. Dividing the score by the ensemble's own uncertainty
$\hat\sigma(x)$ (**normalised** or locally adaptive conformal) gives intervals that are narrow for polar and wide for
lipophilic molecules, and the coverage becomes even (0.885–0.888 in the three groups). The guarantee is still only marginal,
but a good score function brings conditional coverage closer to it. (Conformalised quantile regression, Romano *et al.*
2019, is a popular alternative.)

**When exchangeability breaks.** Calibration molecules came from the training chemistry; what if the test molecules do not?

In [ ]:
# ── 6.3 Coverage under shift, and weighted conformal prediction (Tibshirani et al. 2019) ──
def weighted_conformal_quantile(scores, w_cal, w_test, alpha):
    """Quantile of the calibration scores re-weighted by w = p_test(x)/p_train(x); the test point's weight goes to +∞."""
    order = np.argsort(scores)
    p = np.r_[w_cal[order], w_test] / (w_cal.sum() + w_test)
    cum = np.cumsum(p[:-1])
    k = np.searchsorted(cum, 1 - alpha)
    return np.inf if k >= len(scores) else scores[order][k]


q_ref = conformal_quantile(resid[cal], ALPHA)
print(f"plain conformal coverage: in distribution {(resid[test] <= q_ref).mean():.3f}, "
      f"new scaffolds {(resid[te_scaf] <= q_ref).mean():.3f}, logP > 3 {(resid[te_logp] <= q_ref).mean():.3f}")

# a covariate shift with overlap: the next project favours lipophilic molecules (sampling ∝ exp(1.5 · logP))
desc_cols = slice(X_mol.shape[1] - 6, None)  # the six descriptors
rng = np.random.default_rng(SEED + 63)
cov_plain, cov_weighted = [], []
for _ in range(200):
    perm = rng.permutation(pool_m)
    cal, rest = perm[:N_CAL], perm[N_CAL:]
    p_pick = np.exp(1.5 * logp[rest]); p_pick /= p_pick.sum()
    test = rng.choice(rest, 150, replace=False, p=p_pick)  # shifted test set
    dom = LogisticRegression(max_iter=1000).fit(X_mol[np.r_[cal, test]][:, desc_cols], np.r_[np.zeros(len(cal)), np.ones(len(test))])
    c_cal, c_te = (dom.predict_proba(X_mol[i][:, desc_cols])[:, 1] for i in (cal, test))
    w_cal, w_te = c_cal / (1 - c_cal), c_te / (1 - c_te)  # odds ∝ p_test(x) / p_cal(x)
    cov_plain.append((resid[test] <= conformal_quantile(resid[cal], ALPHA)).mean())
    cov_weighted.append(np.mean([resid[j] <= weighted_conformal_quantile(resid[cal], w_cal, wj, ALPHA)
                                 for j, wj in zip(test, w_te)]))
print(f"🧪 lipophilic-shifted test sets (200 repeats): plain conformal coverage {np.mean(cov_plain):.3f}, "
      f"weighted conformal {np.mean(cov_weighted):.3f}  (nominal {1 - ALPHA:.2f})")

🔍 **What just happened?** For molecules of the training chemistry the plain conformal coverage is 0.88; for new scaffolds it
collapses to 0.06 and for lipophilic extrapolation to 0.38. **The guarantee holds only when calibration and test data are
exchangeable**, and an unseen scaffold or an extrapolated property breaks that.

Some shifts can be repaired. Tibshirani *et al.* (2019) showed that under **covariate shift** — $p(x)$ changes, $p(y\mid x)$
does not — the guarantee is restored by **re-weighting** the calibration scores by the likelihood ratio
$w(x)=p_{\text{test}}(x)/p_{\text{train}}(x)$:

$$\hat q(x)=\text{the }(1-\alpha)\text{ quantile of }\ \sum_{i=1}^{n}\frac{w(x_i)}{\sum_j w(x_j)+w(x)}\,\delta_{s_i}
\;+\;\frac{w(x)}{\sum_j w(x_j)+w(x)}\,\delta_{+\infty}.$$

*In words:* calibration molecules that resemble the test population count more, unrepresentative ones count less, and the
test point itself contributes a small weight at "infinitely wrong" as a safety margin. The ratio $w(x)$ is unknown, but a
domain classifier (Section 3) estimates it: if $c(x)$ is the predicted probability that $x$ came from the test set, then
$w(x)\propto c(x)/(1-c(x))$. In our lipophilic-shifted scenario (the next project favours greasy molecules, whose
measurements are noisier) plain conformal under-covers (0.82) and weighted conformal is back at the nominal level (0.91). It cannot rescue
the new scaffolds: when the supports do not overlap (domain AUROC ≈ 1) the weights are infinite — there is simply no
calibration information about those molecules.

**Conformal prediction sets for classification.** The same recipe gives *sets* of labels. Use the score
$s=1-\hat p_y(x)$ (one minus the probability of the true class) on calibration cells, find $\hat q$, and predict the set
$\{k:\ 1-\hat p_k(x)\le\hat q\}$ — every cell type whose probability is at least $1-\hat q$. Confident cells get one label;
ambiguous ones (CD4 or CD8?) get two.

In [ ]:
# ── 6.4 Conformal prediction sets for cell types; coverage per class and under batch shift ──
P_cal, P_te, P_b2 = (softmax_T(logits[i], T_fit) for i in (cal_c, te_c, b2))
ALPHA_SET = 0.05  # 95 % sets for annotation
q_set = conformal_quantile(1 - P_cal[np.arange(len(cal_c)), y_type[cal_c]], ALPHA_SET)
sets_te, sets_b2 = (1 - P_te) <= q_set, (1 - P_b2) <= q_set
covered = lambda S, y: S[np.arange(len(y)), y]
# recalibration with 200 labelled cells from the new batch (an expert annotates a small sample)
rng = np.random.default_rng(SEED + 64)
lab_b2 = rng.choice(len(b2), 200, replace=False); rest_b2 = np.setdiff1d(np.arange(len(b2)), lab_b2)
q_re = conformal_quantile(1 - P_b2[lab_b2, y_type[b2][lab_b2]], ALPHA_SET)
sets_re = (1 - P_b2[rest_b2]) <= q_re
rows = [("batch-1 test", covered(sets_te, y_type[te_c]).mean(), sets_te.sum(1).mean()),
        ("batch 2, batch-1 calibration", covered(sets_b2, y_type[b2]).mean(), sets_b2.sum(1).mean()),
        ("batch 2, recalibrated on 200 cells", covered(sets_re, y_type[b2][rest_b2]).mean(), sets_re.sum(1).mean())]
print(f"α = {ALPHA_SET}: q̂ = {q_set:.3f}, so a cell type enters the set if its probability ≥ {1 - q_set:.3f}")
print("set sizes in batch-1 test cells (0, 1, 2, 3, …):", np.bincount(sets_te.sum(1)))
print(pd.DataFrame(rows, columns=["cells", "coverage", "mean set size"]).round(3).to_string(index=False))
pool_c, resplit = np.r_[cal_c, te_c], []  # is this split typical? re-split calibration/test 200 times
P_pool = softmax_T(logits[pool_c], T_fit)
s_pool = 1 - P_pool[np.arange(len(pool_c)), y_type[pool_c]]
for _ in range(200):
    pp = rng.permutation(len(pool_c))
    resplit.append((s_pool[pp[len(cal_c):]] <= conformal_quantile(s_pool[pp[:len(cal_c)]], ALPHA_SET)).mean())
print(f"batch-1 coverage over 200 random calibration/test re-splits: mean {np.mean(resplit):.3f} "
      f"(5th–95th percentile {np.percentile(resplit, 5):.3f}–{np.percentile(resplit, 95):.3f})")
per_class = pd.DataFrame({"batch-1 test": [covered(sets_te, y_type[te_c])[y_type[te_c] == k].mean() for k in range(6)],
                          "batch 2": [covered(sets_b2, y_type[b2])[y_type[b2] == k].mean() for k in range(6)],
                          "batch 2 recalibrated": [covered(sets_re, y_type[b2][rest_b2])[y_type[b2][rest_b2] == k].mean()
                                                   for k in range(6)]}, index=TYPES)
print(per_class.round(2).T.to_string())
fig, ax = plt.subplots(figsize=(10, 3.2))
for k, (col, c) in enumerate(zip(per_class.columns, [OKABE_ITO[0], OKABE_ITO[1], OKABE_ITO[2]])):
    ax.bar(np.arange(6) + (k - 1) * 0.27, per_class[col], 0.27, color=c, label=col)
ax.axhline(1 - ALPHA_SET, color="k", ls="--", lw=1); ax.set_ylim(0, 1.05)
ax.set_xticks(range(6)); ax.set_xticklabels(TYPES); ax.set_ylabel("coverage of the true type")
ax.set_title("SIMULATED: 95 % conformal sets — fine on average, uneven across cell types, broken by an uncorrected batch")
ax.legend(fontsize=7, loc="center left", bbox_to_anchor=(1.0, 0.5)); plt.tight_layout(); plt.show()

🔍 **What just happened?** With $\alpha=0.05$ a cell type enters the set if its probability is at least 0.35. On batch-1
test cells 1,153 of 1,188 sets contain a single label, 32 contain two (mostly CD4 *or* CD8 T) and 3 are empty (no type is
plausible enough — an honest "I don't know"); the average size is 1.02. The coverage on this test split is 0.925, a little
below the nominal 0.95. Is the guarantee broken? Re-splitting the same calibration and test cells 200 times gives a mean
coverage of 0.952 (5th–95th percentile 0.935–0.967): the guarantee is an average over calibration sets, and our particular
split is an unlucky one. (Part of the shortfall may come from having fitted the temperature on the same calibration cells —
strictly, conformal calibration data should not be used to fit anything else.)

Per class, coverage is uneven: monocytes, B and dendritic cells are covered 97–100 % of the time, NK cells 92 %, CD4 and CD8
T cells only 84–85 % — the marginal guarantee is paid for by the easy classes. **Mondrian** (class-conditional) conformal prediction fixes
this by computing a separate $\hat q$ for each class (✏️ below). Under the batch shift the batch-1 calibration no longer
applies: overall coverage drops to 0.71 — and to 0.36 for CD8 T cells and 0.47 for B cells, while CD4 T cells are covered
99 % of the time (batch 2 pushes many cells towards "CD4-like"). Annotating just 200 cells of the new batch and
recalibrating restores the overall coverage (0.94) at the price of much larger sets (2.6 labels on average): larger sets are
the honest price of an uncorrected batch, and a signal that batch correction ([Module 31](31_batch_integration.ipynb)) is
needed.

⚠️ **What the guarantee does and does not promise.** It promises that, *on average over exchangeable test cases*, the set
or interval contains the truth with probability ≥ $1-\alpha$. It does **not** promise coverage for each patient, each
subgroup or a shifted population, and it does not make a poor model good — a poor model simply gets wide intervals or large
sets. That is still valuable: the size of the set is an honest, model-agnostic measure of how hard each case is.

For a longer, clinically oriented treatment — adaptive prediction sets, subgroup audits, shift stress tests — see the
repository's conformal-prediction tutorial: [`../../ConformalPrediction/`](../../ConformalPrediction/).

✏️ **Your turn.** Implement Mondrian conformal sets: compute $\hat q_k$ from the calibration cells of class $k$ only and
include class $k$ in the set when $1-\hat p_k\le\hat q_k$. Check that the per-class coverage on batch-1 test cells is now
close to 0.90 for every type. What happens to the average set size?

### 📄 Research connection — Conformal prediction in biology and medicine

- **What the field does.** Conformal prediction was developed by Vovk, Gammerman & Shafer (2005) as a general way to attach valid
    confidence to any predictor; Lei *et al.* (2018) analysed split conformal regression in the modern statistical setting,
    Romano *et al.* (2019) made intervals adaptive with quantile regression, and Tibshirani *et al.* (2019) extended the
    guarantee to covariate shift by weighting. Angelopoulos & Bates (2023) wrote the standard tutorial. In biology it has
    been applied to drug–target and toxicity prediction (where it has a long history in cheminformatics), to cell-type
    annotation, and to protein design.
- **What this notebook builds.** Split conformal intervals around a deep ensemble (constant and normalised scores), a 500-split check of the
    coverage guarantee against its Beta distribution, weighted conformal prediction with a domain classifier, and conformal
    prediction sets for cell types with recalibration after a batch shift. Production libraries: MAPIE (scikit-learn
    compatible) and crepes in Python; the repository notebook [`../../ConformalPrediction/`](../../ConformalPrediction/) adds
    adaptive prediction sets and subgroup audits.
- **Where the frontier is.** **Design** creates a special shift: the test sequences are *chosen by* the model itself, so they are not
    exchangeable with the training data. Fannjiang *et al.* (2022) derived conformal intervals valid under this "feedback
    covariate shift" for protein design. Other active directions: conditional and group-conditional guarantees, conformal
    risk control (bounding expected losses other than miscoverage), conformal methods for time series and for outputs of
    generative models and LLMs, and how to communicate sets and intervals to clinicians.
- **Read next.** Angelopoulos & Bates 2023, *Conformal prediction: a gentle introduction* (Found Trends Mach Learn);
    Tibshirani *et al.* 2019, *Conformal prediction under covariate shift* (NeurIPS); Fannjiang *et al.* 2022, *Conformal
    prediction under feedback covariate shift for biomolecular design* (PNAS).

---
## 7 · Interpretability pitfalls

> 📍 **Where we are.** A trustworthy model is not only accurate and calibrated; its explanations — "these genes matter",
> "these bases drive activity" — are often the scientific result. Explanations can fail as silently as accuracy. Two classic
> traps, each checked against the truth.

**(a) Correlated features share (and inflate) importance.** Genes come in co-regulated modules. We simulate 12 genes for 600
patients: gene 0 and gene 3 cause the disease (log-odds $2x_0+x_3$); genes 1 and 2 are near-copies of gene 0 (correlation
0.95) with **no effect of their own**; the other eight are noise. We compare three importance measures for a random forest
([Module 10](10_decision_trees_random_forests.ipynb)): the impurity-based importance (MDI, scikit-learn's default), the
permutation importance on held-out patients, and a *group* importance — the drop in AUROC when a whole group of correlated
genes is removed and the forest refitted.

In [ ]:
# ── 7.1 Random-forest importance with correlated genes (SIMULATED) ──
rng = np.random.default_rng(SEED + 71)
n = 600
G7 = rng.normal(size=(n, 12))
G7[:, 1] = 0.95 * G7[:, 0] + np.sqrt(1 - 0.95 ** 2) * rng.normal(size=n)  # near-copies of gene 0 (no own effect)
G7[:, 2] = 0.95 * G7[:, 0] + np.sqrt(1 - 0.95 ** 2) * rng.normal(size=n)
y7 = (rng.random(n) < sigmoid(2 * G7[:, 0] + G7[:, 3])).astype(int)  # TRUE causes: genes 0 and 3
tr7, te7 = np.arange(400), np.arange(400, n)
forest = RandomForestClassifier(300, random_state=SEED, n_jobs=1).fit(G7[tr7], y7[tr7])
base_auc = roc_auc_score(y7[te7], forest.predict_proba(G7[te7])[:, 1])
perm_imp = permutation_importance(forest, G7[te7], y7[te7], n_repeats=10, random_state=SEED, scoring="roc_auc")


def drop_group_importance(cols):
    """AUROC lost when the genes in `cols` are removed and the forest is refitted."""
    keep = [j for j in range(12) if j not in cols]
    f = RandomForestClassifier(300, random_state=SEED, n_jobs=1).fit(G7[tr7][:, keep], y7[tr7])
    return base_auc - roc_auc_score(y7[te7], f.predict_proba(G7[te7][:, keep])[:, 1])


imp = pd.DataFrame({"MDI (impurity)": forest.feature_importances_, "permutation (ΔAUROC)": perm_imp.importances_mean},
                   index=[f"gene {j}" for j in range(12)])
print("SIMULATED: genes 0 and 3 are causal; genes 1, 2 are near-copies of gene 0")
print(imp.round(3).T.to_string())
print(f"\nforest AUROC {base_auc:.3f}; change in AUROC when the forest is refitted without ...")
for cols in ([0], [1, 2], [0, 1, 2], [3]):
    print(f"   genes {cols}: {-drop_group_importance(cols):+.3f}")

🔍 **What just happened?** The truth is "genes 0 and 3". MDI ranks gene 0 first (0.18) but gives its two inert copies 0.17
and 0.13 — more than gene 3 (0.11), a real cause with half of gene 0's effect — and even pure-noise genes get about 0.05 each
(MDI is biased towards continuous features with many split points). Permutation importance is more honest about noise
(≈ 0), but it **under-states** gene 0 (0.07, barely more than gene 3's 0.06, although its effect is twice as large) and
still credits the copies (0.02–0.03): when gene 0 is shuffled, the forest reads its copies instead. The refitting experiment
makes the redundancy explicit: removing gene 0 alone (−0.008) or its two copies (+0.003) costs nothing, removing gene 3
costs 0.05, and removing all three of the correlated genes
costs 0.21 AUROC. **Importance is a property of the model and the feature set, not of the biology**: the forest cannot know
which of three correlated genes is causal — no observational model can (that needs interventions, [Module 41](41_perturbation_causal_inference.ipynb)).
Report importance for groups of correlated features, or use conditional importance (Strobl *et al.* 2008).

**(b) Saliency maps need sanity checks.** For sequence models a *saliency map* scores each base by the gradient of the
prediction with respect to the input, times the input ("gradient × input", [Module 36](36_regulatory_genomics_deep_learning.ipynb)).
Adebayo *et al.* (2018) proposed a simple test: **randomise the model's weights** (starting with the top layer) and recompute
the map. If the map barely changes, it is explaining the *input*, not the *model* — however plausible it looks. We train a
small CNN on SIMULATED enhancer sequences, where we know the true motif positions.

In [ ]:
# ── 7.2 A small CNN on SIMULATED enhancers, and saliency under cascading randomisation ──
class MotifCNN(nn.Module):
    """16 convolution filters of width 11 → mean and max pooling → linear output (activity)."""

    def __init__(self):
        super().__init__()
        self.conv, self.out = nn.Conv1d(4, 16, 11, padding=5), nn.Linear(32, 1)

    def forward(self, x):
        h = F.relu(self.conv(x))
        return self.out(torch.cat([h.mean(-1), h.amax(-1)], 1)).squeeze(-1)


enh = simulate_regulatory_sequences(np.random.default_rng(SEED + 72), n_seqs=3000, length=150, motif_rate=2.0)
X_enh = torch.tensor(np.array([one_hot_dna(s) for s in enh["sequences"]]), dtype=torch.float32).permute(0, 2, 1)
y_enh = torch.tensor((enh["activity"] - enh["activity"].mean()) / enh["activity"].std(), dtype=torch.float32)
torch.manual_seed(SEED)
cnn, t0 = MotifCNN(), time.time()
opt = torch.optim.Adam(cnn.parameters(), lr=3e-3)
for epoch in range(12):
    perm = torch.randperm(2500)
    for k in range(0, 2500, 64):
        b = perm[k:k + 64]
        loss = F.mse_loss(cnn(X_enh[b]), y_enh[b]); opt.zero_grad(); loss.backward(); opt.step()


def saliency(model, x):
    """|gradient × input| summed over the 4 bases: one importance score per position."""
    x = x.clone().requires_grad_(True)
    model(x).sum().backward()
    return (x.grad * x).sum(1).abs().detach().numpy()


def motif_mask(seq):
    """True at every position covered by a motif site (the ground truth for attribution)."""
    m = np.zeros(len(seq), bool)
    for consensus, _ in REGULATORY_MOTIFS.values():
        for s in find_motif_sites(seq, consensus):
            m[s:s + len(consensus)] = True
    return m


top_random = copy.deepcopy(cnn); nn.init.normal_(top_random.out.weight, 0, 0.1)  # randomise the top layer only
all_random = copy.deepcopy(cnn)
for prm in all_random.parameters():
    nn.init.normal_(prm, 0, 0.1)  # randomise every layer
ids = np.arange(2500, 2800)  # held-out sequences
masks = np.array([motif_mask(enh["sequences"][i]) for i in ids])
maps = {name: saliency(m, X_enh[ids]) for name, m in [("trained", cnn), ("top layer randomised", top_random),
                                                      ("all layers randomised", all_random)]}
rows = []
for name, m in [("trained", cnn), ("top layer randomised", top_random), ("all layers randomised", all_random)]:
    with torch.no_grad():
        r_pred = np.corrcoef(m(X_enh[ids]).numpy(), y_enh[ids].numpy())[0, 1]
    auc_sites = np.mean([roc_auc_score(mk, s) for s, mk in zip(maps[name], masks) if 0 < mk.sum() < len(mk)])
    rho = np.mean([stats.spearmanr(s, t)[0] for s, t in zip(maps[name], maps["trained"])])
    rows.append((name, r_pred, auc_sites, rho))
print(f"(CNN trained in {time.time() - t0:.1f} s)")
print(pd.DataFrame(rows, columns=["model", "r(prediction, activity)", "saliency AUROC for motif positions",
                                  "rank corr. with trained map"]).round(3).to_string(index=False))

In [ ]:
# ── 7.3 One sequence, three saliency maps ──
k = int(np.argmax(masks.sum(1) * (masks.sum(1) < 40)))  # a held-out sequence with several motif sites
fig, axes = plt.subplots(3, 1, figsize=(12, 4.6), sharex=True)
for ax, (name, M) in zip(axes, maps.items()):
    ax.fill_between(np.arange(150), 0, masks[k] * M[k].max(), color=OKABE_ITO[3], alpha=0.3, step="mid",
                    label="true motif sites")
    ax.plot(M[k], color=OKABE_ITO[0], lw=1.2)
    ax.set_ylabel("|grad × input|", fontsize=8); ax.set_title(name, fontsize=9, loc="left")
axes[0].legend(loc="upper right", fontsize=7); axes[-1].set_xlabel("position in the SIMULATED enhancer")
plt.suptitle("saliency still 'finds the motifs' after the output layer is randomised — it explains the filters, not the prediction",
             y=1.0); plt.tight_layout(); plt.show()

🔍 **What just happened?** The trained CNN predicts activity on held-out sequences reasonably well (r = 0.68), and its
saliency map lights up the true motif positions (AUROC 0.75). Now the sanity check. After **randomising only the output
layer**, the model's predictions are meaningless (r = −0.14) — yet the saliency map is largely unchanged (rank correlation
0.76 with the trained map) and still "finds the motifs" (AUROC 0.70). The map reflects which positions the convolution
filters respond to; whether the model *uses* those responses to predict activity is lost when we take absolute values. Only
when every layer is randomised does the map become noise (AUROC 0.50). A map that survives randomisation of the layer that
makes the decision cannot be evidence about how the decision is made.

⚠️ **The general lesson.** "The explanation agrees with what biologists expected" is not validation — a method that ignored
the model could agree too. Validate attribution methods with randomisation tests and, where possible, with ground truth (in
silico mutagenesis, synthetic motifs planted in known places, as here), and treat explanations as **hypotheses to test**.

✏️ **Your turn.** Repeat the comparison with the *signed* saliency (remove `.abs()` in `saliency`). Does the top-layer-
randomised map still correlate with the trained one? What does this tell you about reporting absolute attributions?

---
## 8 · Fairness and equity: does the model work for everyone?

> 📍 **Where we are.** A model can be well calibrated and honestly evaluated *on average* and still fail for a group that is
> rare in the training data. In genetics and medicine those groups are often defined by ancestry, because research cohorts
> have been dominated by people of European ancestry. [Module 35](35_statistical_genetics.ipynb) showed that polygenic
> scores lose accuracy across ancestries because linkage disequilibrium differs (Martin *et al.* 2019 found scores roughly
> two to four and a half times less accurate in East Asian and African-ancestry individuals). Here we look at a second, very
> general mechanism with a diagnostic classifier.

**The scenario (SIMULATED).** A disease raises three blood biomarkers. In population B a genetic variant — when inherited from
both parents — lowers the *healthy* level of biomarker 0 without any disease (real examples of this kind exist: the
Duffy-null variant, common in people of African ancestry, is associated with lower neutrophil counts, which led to
reference ranges flagging healthy people as abnormal). The variant is rare in population A. The disease prevalence is the
same (30 %) in both populations. The training cohort is 95 % population A. We draw the two populations' genomes with
`simulate_genotypes` and take as "the variant" the SNP whose allele frequency differs most between them.

In [ ]:
# ── 8.1 A biomarker that behaves differently in two populations (SIMULATED) ──
rng = np.random.default_rng(SEED + 81)
N8 = 12000
geno, popn, freqs = simulate_genotypes(rng, N8, 300, n_pops=2, fst=0.15)
v = int(np.argmax(np.abs(freqs[0] - freqs[1])))  # most differentiated SNP
allele_b = int(freqs[1, v] > freqs[0, v])  # the allele that is common in population B
copies = geno[:, v] if allele_b == 1 else 2 - geno[:, v]
variant = (copies == 2).astype(float)  # homozygous carriers
disease = (rng.random(N8) < 0.30).astype(int)
bio = rng.normal(size=(N8, 5))
bio[:, :3] += disease[:, None] * np.array([1.0, 0.8, 0.6])  # disease raises biomarkers 0–2
bio[:, 0] -= 1.2 * variant  # the variant lowers biomarker 0 in health and disease
X8 = np.c_[bio, variant]
print(f"SIMULATED SNP {v}: frequency of the B-common allele {freqs[0, v] if allele_b else 1 - freqs[0, v]:.2f} in A, "
      f"{freqs[1, v] if allele_b else 1 - freqs[1, v]:.2f} in B")
print(f"homozygous carriers: {variant[popn == 0].mean():.1%} of population A, {variant[popn == 1].mean():.1%} of population B")

In [ ]:
# ── 8.2 Subgroup report for four modelling choices ──
idx_a, idx_b = rng.permutation(np.flatnonzero(popn == 0)), rng.permutation(np.flatnonzero(popn == 1))
test_a, test_b, pool_a, pool_b = idx_a[:2000], idx_b[:2000], idx_a[2000:], idx_b[2000:]


def subgroup_report(frac_b, cols, n_train=4000):
    """Train a logistic regression on n_train people (fraction frac_b from B) using columns `cols`; report per group."""
    n_b = int(n_train * frac_b)
    tr = np.r_[pool_a[:n_train - n_b], pool_b[:n_b]]
    m = LogisticRegression(max_iter=1000).fit(X8[tr][:, cols], disease[tr])
    out = {}
    for g, te in [("A", test_a), ("B", test_b)]:
        p = m.predict_proba(X8[te][:, cols])[:, 1]; call = p > 0.5; y_ = disease[te]
        out[g] = dict(acc=(call == y_).mean(), TPR=call[y_ == 1].mean(), FPR=call[y_ == 0].mean(), mean_p=p.mean())
    return out


bio_cols, all_cols = [0, 1, 2, 3, 4], [0, 1, 2, 3, 4, 5]
strategies = [("95 % A, biomarkers", 0.05, bio_cols), ("50 % A, biomarkers", 0.5, bio_cols),
              ("95 % A, + variant", 0.05, all_cols), ("50 % A, + variant", 0.5, all_cols)]
fair = {name: subgroup_report(f, c) for name, f, c in strategies}
table = pd.DataFrame({(name, g): fair[name][g] for name in fair for g in ("A", "B")}).T
overall = {name: (fair[name]["A"]["acc"] * 0.95 + fair[name]["B"]["acc"] * 0.05) for name in fair}
print(table.round(3).to_string())
print("\noverall accuracy in a deployment population that is 95 % A:", {k: round(v, 3) for k, v in overall.items()})

In [ ]:
# ── 8.3 Who is missed? Sensitivity (TPR) per population ──
fig, ax = plt.subplots(figsize=(8.5, 3.3))
x = np.arange(len(strategies)); w = 0.35
for k, (g, c) in enumerate([("A", OKABE_ITO[0]), ("B", OKABE_ITO[1])]):
    vals = [fair[name][g]["TPR"] for name, _, _ in strategies]
    ax.bar(x + (k - 0.5) * w, vals, w, color=c, label=f"population {g}")
    for xi, val in zip(x, vals):
        ax.text(xi + (k - 0.5) * w, val + 0.01, f"{val:.2f}", ha="center", fontsize=7)
ax.set_xticks(x); ax.set_xticklabels([s[0] for s in strategies], fontsize=8); ax.set_ylim(0, 0.75)
ax.set_ylabel("sensitivity (true-positive rate)"); ax.legend(fontsize=8)
ax.set_title("SIMULATED: the same test misses more diseased people in population B until the mechanism is modelled")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Trained on a 95 %-A cohort with biomarkers only, the classifier detects 57 % of diseased people in
population A but only 31 % in population B: the variant pulls biomarker 0 down, so diseased carriers look healthy. Overall
accuracy (0.81) and even accuracy within B (0.77, because 70 % of people are healthy and correctly called healthy) hide this
almost completely — **only a subgroup report of error types shows who is missed**. Simply collecting a balanced cohort
(50 % B) helps B only partly (sensitivity 0.41) and costs A more false positives (0.135 instead of 0.088), because a single
set of biomarker weights must compromise between two biologies. Adding the **mechanism** — the genotype at the variant —
lets the model learn a population-independent rule, and the gap almost closes (0.57 vs 0.54 with the 95 %-A cohort, 0.60 vs
0.52 with the balanced one).

Three lessons that carry over to real data: (1) report performance, calibration and error types **per subgroup**, not only
overall; (2) diverse training data is necessary but not sufficient — the model must be able to represent *why* groups differ;
(3) prefer measured biological mechanisms to coarse proxies. Ancestry or self-reported race is a weak proxy for a specific
variant, mixes biology with social and environmental factors, and has been removed from some clinical equations — e.g. US
kidney societies recommended race-free equations for kidney function in 2021 (Inker *et al.* 2021).

⚠️ **There is no single definition of "fair".** Equal sensitivity, equal false-positive rates and equal calibration across
groups cannot in general all hold at once when prevalence differs between groups; choosing which to prioritise is an ethical
decision that should be made explicitly, with the affected communities, not by default.

✏️ **Your turn.** Keep the "95 % A, biomarkers" model but choose a separate decision threshold for each population so that
both have a sensitivity of 0.5 (search the thresholds on a validation split). What happens to the false-positive rates? Is
this better or worse than adding the variant, and what information does each fix require at deployment?

---
## 9 · Reproducibility and reporting

> 📍 **Where we are.** A result can be trusted only if others can reproduce it and know its limits. This section collects
> the habits that make that possible: fixed seeds, recorded versions, variability across seeds, and structured reports —
> datasheets for data sets (Gebru *et al.* 2021), model cards for models (Mitchell *et al.* 2019) and DOME for papers.

In [ ]:
# ── 9.1 Seeds, versions and the variability that a single run hides ──
import sklearn
import scipy
print("versions:", f"Python {sys.version.split()[0]}, NumPy {np.__version__}, SciPy {scipy.__version__}, "
      f"pandas {pd.__version__}, scikit-learn {sklearn.__version__}, PyTorch {torch.__version__}")
rerun_mu, _ = train_mean_variance(X_mol, y_mol, tr_m, seed=SEED)  # same seed as ensemble member 0
print(f"re-running member 0 with the same seed reproduces it exactly: {np.allclose(rerun_mu, MU[0])}")
seed_rmse = np.sqrt(((MU[:, pool_m] - y_true_mol[pool_m]) ** 2).mean(1))
print(f"in-distribution RMSE of the 5 members (5 seeds): {np.round(seed_rmse, 3)} "
      f"→ {seed_rmse.mean():.3f} ± {seed_rmse.std(ddof=1):.3f}; ensemble mean: "
      f"{np.sqrt(((mu_ens[pool_m] - y_true_mol[pool_m]) ** 2).mean()):.3f}")

In [ ]:
# ── 9.2 A model card for the cell-type classifier, filled with this notebook's own numbers ──
cov_b1 = covered(sets_te, y_type[te_c]).mean()
card = {
    "Model": "CellTypeMLP — 50 PCs → 2 × 128 ReLU → 6 PBMC types; temperature-scaled; 95 % conformal sets",
    "Intended use": "teaching; annotating PBMC-like cells from the SAME protocol and batch as training",
    "Not intended for": "other tissues, other protocols or batches without recalibration; any clinical decision",
    "Training data": f"SIMULATED (simulate_scrna), batch 1 only, {len(tr_c)} cells, balanced types, ≈120 UMIs/cell",
    "Evaluation": f"batch-1 test accuracy {calib.iloc[0, 1]:.3f}; batch 2 (unseen batch) {calib.iloc[1, 1]:.3f}",
    "Calibration": f"ECE {calib.iloc[0, 3]:.3f} → {calib.iloc[0, 5]:.3f} after temperature scaling (T = {T_fit:.2f}); "
                   f"batch 2: {calib.iloc[1, 5]:.3f}",
    "Uncertainty": f"conformal sets: coverage {cov_b1:.3f}, mean size {sets_te.sum(1).mean():.2f} (batch 1); "
                   f"{covered(sets_b2, y_type[b2]).mean():.3f} in batch 2 without recalibration",
    "Subgroups": "coverage per type (batch 1): " + ", ".join(f"{t} {c:.2f}" for t, c in per_class['batch-1 test'].items()),
    "Known limitations": "CD4/CD8 T confusion; no batch correction; over-confident before scaling; label shift biases counts",
    "Reproducibility": f"SEED = {SEED}; versions printed in cell 9.1; code = this notebook",
}
for key, value in card.items():
    print(f"{key:18s} | {value}")

🔍 **What just happened?** Re-running with the same seed reproduced the network exactly, but the five seeds give different
models whose in-distribution RMSE ranges from 0.13 to 0.16 (0.148 ± 0.009): a single run would have reported one of these numbers as if it
were *the* performance. Report the spread over seeds (and over data splits) — and note that the ensemble mean beats every
member. The model card is generated from computed values, so it cannot drift out of date when the code changes; it states
what the model is for, what it must not be used for, how it was evaluated — including under shift and per subgroup — and
its known failure modes.

✏️ **Your turn — DOME for an earlier module.** Pick [Module 40](40_molecular_ml_drug_discovery.ipynb) (molecular ML) or
[Module 30](30_single_cell_rnaseq.ipynb) (single-cell) and write four short paragraphs: **D**ata (source, size, how it was
split and how independent the splits are), **O**ptimisation (preprocessing, feature selection, tuning — inside or outside the
training folds?), **M**odel (what, why, availability), **E**valuation (metrics, baselines, uncertainty, the claim the test
set supports). Name one leak or shift that the module's evaluation could still be vulnerable to.

---
## 10 · 🧭 From toy to real data

> 📍 **Where we are.** Every experiment above knew the truth. Real projects do not — which is exactly why the habits
> matter. Here is what changes, without downloads.

**Hidden structure is the norm.** In real data you must *discover* the groups that cause leakage: related individuals
(kinship from genotypes, as in UK Biobank's relatedness tables), repeated samples of one patient, homologous proteins
(cluster with [MMseqs2](https://github.com/soedinglab/MMseqs2) or CD-HIT at a stated identity threshold), shared chemical
scaffolds, shared labs or plates. Benchmarks increasingly ship fixed, documented splits: the
[Therapeutics Data Commons](https://tdcommons.ai) offers scaffold and time splits for molecules, and
[ProteinGym](https://proteingym.org) evaluates variant-effect predictors across many assays. Single-cell atlases such as
[CELLxGENE](https://cellxgene.cziscience.com) record donor, assay and batch metadata — use them to split by donor and study.

**Shift is the norm too.** New sites, protocols, reagent lots and populations. Run a domain classifier against the
training data before trusting predictions; recalibrate (temperature or conformal) with a small labelled sample from each new
site; monitor performance after deployment. Biobanks with diverse participants — [All of Us](https://allofus.nih.gov),
[UK Biobank](https://www.ukbiobank.ac.uk), [H3Africa](https://h3africa.org) — make subgroup evaluation in genetics possible.

**Tools.** scikit-learn (`GroupKFold`, `StratifiedGroupKFold`, `TimeSeriesSplit`, `CalibratedClassifierCV`,
`permutation_importance`); MAPIE and crepes for conformal prediction; Fairlearn for subgroup metrics; Chemprop for molecular
ensembles. Compute is rarely the bottleneck: these checks cost seconds to minutes, far less than training the model.

**The first three things that go wrong.**

1. **An unrecognised group.** Two "independent" data sets turn out to share patients, cell lines or near-identical
   sequences; or a confounder (site, batch, time) is aligned with the label. Look for it *before* modelling: plot the data
   by every metadata column and run a domain classifier between your splits.
2. **A test set that answers the wrong question.** The split must match the claim ("new patients", "new families", "new
   sites", "next year"). State the claim first, then design the split — and keep a truly external test set untouched until
   the end (Module 23's nested CV; [Module 44](44_capstone_project.ipynb)'s sealed test set).
3. **Uncertainty that is not checked.** Probabilities and intervals from a model are claims that must be calibrated and
   tested for coverage, per subgroup and under shift. Conformal prediction makes coverage cheap to guarantee in-distribution;
   nothing makes it automatic out of distribution.

## 📌 Key takeaways

- **Leakage** makes test sets too similar to training sets. Split by the unit of deployment — donor, family, scaffold, time —
  and keep every label-using step inside the training folds; in our simulations naive estimates exceeded the truth in all
  four cases (e.g. 0.91 vs 0.75 accuracy for donors, R² 0.56 vs 0.28 for sequence families).
- **Shortcuts** (site signatures) can make a model look better than the true signal allows and then fail at the next site.
  The label itself can be biased (Obermeyer) — no split detects that.
- **Shift** comes as covariate shift, label shift and concept drift. A domain classifier detects covariate shift cheaply but
  can miss label shift; EM re-estimates class proportions and corrects posteriors for label shift.
- **Calibration**: over-parameterised networks are over-confident; temperature (multi-class) and Platt (binary) scaling fix it
  on a calibration set without changing accuracy — and calibration degrades again under shift.
- **Uncertainty**: total variance = aleatoric + epistemic. A deep ensemble tracked the true noise but under-estimated it, and
  its epistemic part barely noticed unseen scaffolds — "unsure" is not a reliable alarm out of distribution.
- **Conformal prediction** turns any model into intervals or sets with a finite-sample guarantee
  $P(Y\in\hat C(X))\ge1-\alpha$ under exchangeability; verified over 500 splits. The guarantee is marginal: check subgroups,
  use adaptive scores, weight for covariate shift, recalibrate for new batches.
- **Explanations** need tests: importance is shared among correlated genes, and saliency maps can survive randomisation of
  the layer that makes the decision.
- **Fairness**: report error types per subgroup; diverse data helps most when the model can represent the mechanism that
  differs; write model cards and follow DOME.

---
## ✅ Checkpoint

1. A paper predicts patient response from single-cell profiles of 12 patients (≈ 5,000 cells each) and reports 97 % accuracy with 10-fold cross-validation over cells. What is the problem and how would you fix it?
   <details><summary>Answer</summary>Cells of each patient appear in both training and test folds, so the model can recognise patients (donor fingerprint) instead of learning response. Split by patient (leave-one-patient-out or grouped CV) and, ideally, test on an independent cohort; with 12 patients, report the uncertainty of the estimate.</details>
2. Why does a 1-nearest-neighbour model look good under a random split of a sequence data set with families, and terrible on new families?
   <details><summary>Answer</summary>Under a random split almost every test sequence has a ~95 %-identical sibling in training, whose activity it copies. On new families the nearest neighbour is unrelated, so the copied value is noise — R² can be negative.</details>
3. What does the expected calibration error measure, and why can temperature scaling reduce it without changing accuracy?
   <details><summary>Answer</summary>ECE is the bin-size-weighted average gap between confidence and accuracy across confidence bins. Dividing all logits by T rescales them without changing their order, so the predicted class (and accuracy) stays the same while confidences move.</details>
4. By hand: a split-conformal regression uses n = 19 calibration residuals and α = 0.1. Which order statistic is $\hat q$? What if n = 8?
   <details><summary>Answer</summary>$\lceil (19+1)(0.9)\rceil = 18$: the 18th smallest of 19 residuals. For n = 8, $\lceil 9\cdot0.9\rceil=9>8$, so $\hat q=\infty$: with so few calibration points the only interval guaranteed to cover 90 % is infinite.</details>
5. What does the conformal guarantee promise, and what does it not?
   <details><summary>Answer</summary>It promises $P(Y\in\hat C(X))\ge1-\alpha$ on average over exchangeable calibration and test data, for any model. It does not promise coverage for each individual, each subgroup, or shifted data (e.g. new scaffolds, a new batch), and it does not make a model accurate — a poor model gets wide intervals.</details>
6. Write the decomposition of predictive variance for an ensemble and say which part should grow for a molecule unlike any training molecule.
   <details><summary>Answer</summary>$\mathrm{Var}=\frac1M\sum_m\sigma_m^2+\frac1M\sum_m(\mu_m-\bar\mu)^2$ (aleatoric + epistemic). The epistemic part should grow — but our experiment showed it may grow little (unseen scaffolds), so it cannot be relied on alone.</details>
7. Code reading: in `em_label_shift`, what do the lines `W = P * pi / pi_train` and `pi = W.mean(0)` correspond to?
   <details><summary>Answer</summary>The E-step (Bayes' rule re-weights each class posterior by the ratio of new to training prior; rows are then normalised) and the M-step (the new class proportions are the average adjusted posteriors), exactly as for the mixing weights of a mixture model (Module 13).</details>
8. Why can a random forest's importance be misleading when genes are correlated, and what would you report instead?
   <details><summary>Answer</summary>Splits can use any of several correlated genes, so impurity importance is spread among them (and inflated for non-causal copies), while permutation importance of one gene is small because its copies substitute for it. Report importance for groups of correlated genes (drop-group or grouped permutation) or conditional importance, and do not read importance as causation.</details>

## 🧠 Questions & discussion

1. A model for predicting antibiotic resistance from bacterial genomes is evaluated on isolates collected in the same hospitals and years as the training isolates. List three ways in which this evaluation could leak, and design a split for the claim 'it works for next year's isolates in other countries'.
2. Conformal prediction gives a 90 % guarantee *on average*. For a clinical test, is marginal coverage enough? Which subgroups would you insist on checking, and what would you do if one of them were under-covered?
3. In Section 8 adding a genotype fixed the disparity. Discuss the ethics and practicality of collecting genetic data for a diagnostic test, compared with using self-reported ancestry or simply collecting more diverse data.
4. Domain classifiers detect covariate shift well but missed our label shift. Propose a monitoring dashboard for a deployed cell-type annotator that would catch both.
5. Saliency maps and attention weights are often shown as 'what the model looks at'. After Section 7, what evidence would convince you that an attribution map reflects a real regulatory mechanism?

## 🚀 Bonus — going further

- **Mondrian conformal prediction.** Implement class-conditional prediction sets (✏️ in Section 6) and compare per-class coverage and set size with the marginal version, in batch 1 and in batch 2 after recalibration.
- **Adaptive prediction sets (APS).** Replace the score $1-\hat p_y$ by the cumulative probability of all classes ranked above and including the true one (Romano, Sesia & Candès 2020). Are the sets better adapted to hard cells? The repository tutorial [`../../ConformalPrediction/`](../../ConformalPrediction/) implements APS.
- **Conformalised quantile regression.** Train two networks (or gradient-boosting models) for the 5 % and 95 % quantiles of potency and conformalise them (Romano *et al.* 2019). Compare conditional coverage by noise group with Section 6.2.
- **A model card for a course model.** Write a full model card (and a datasheet for its training data) for the CNN of [Module 36](36_regulatory_genomics_deep_learning.ipynb) or the QSAR model of Module 40, including a subgroup analysis.
- **Real tools.** Re-implement Sections 4 and 6 with scikit-learn's `CalibratedClassifierCV` and the MAPIE library on a public data set, and check that you obtain the same coverage behaviour.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 42 · Trustworthy biological ML
Biological question: do our performance estimates and uncertainties mean what they claim?
Data (simulated): simulate_scrna (donors, batches, cell types), sequence families from simulate_regulatory_sequences,
                  simulate_molecules + known heteroscedastic noise, simulate_genotypes (2 populations), hospitals, drift.
                  Ground truth: fresh cohorts, true noise sd, true causes, true motif sites, true composition.
Method: grouped / cluster / forward-chaining splits; domain classifier; EM label-shift correction; ECE, temperature and
        Platt scaling; deep ensemble; split, normalised and weighted conformal prediction; importance and saliency
        sanity checks; subgroup reporting; model card
Key equation: q̂ = ⌈(n+1)(1−α)⌉-th smallest calibration score ;  Var = E[σ²] + Var[μ] ;  ECE = Σ (n_b/n)|acc − conf|
Key parameters: split unit, calibration-set size n, α, temperature T, ensemble size M, subgroup definitions
Ground-truth check result: naive vs honest vs truth for 4 leaks = … ; conformal coverage over 500 splits = … ;
                           weighted conformal under shift = … ; TPR per population = …
What changed when I modified … (clustering threshold / calibration size / weight decay / training mix): …
One failure mode: (e.g. epistemic uncertainty blind to unseen scaffolds; saliency survives top-layer randomisation)
Connection to research: Walsh 2021 (DOME); Guo 2017; Lakshminarayanan 2017; Tibshirani 2019; tool: MAPIE, Fairlearn
```

## 📚 References

- Walsh I. *et al.* (2021) DOME: recommendations for supervised machine learning validation in biology. *Nat Methods* 18:1122–1127.
- Whalen S., Schreiber J., Noble W. S., Pollard K. S. (2022) Navigating the pitfalls of applying machine learning in genomics. *Nat Rev Genet* 23:169–181.
- Kapoor S., Narayanan A. (2023) Leakage and the reproducibility crisis in machine-learning-based science. *Patterns* 4:100804.
- Park Y., Marcotte E. M. (2012) Flaws in evaluation schemes for pair-input computational predictions. *Nat Methods* 9:1134–1136.
- Zech J. R. *et al.* (2018) Variable generalization performance of a deep learning model to detect pneumonia in chest radiographs: a cross-sectional study. *PLoS Med* 15:e1002683.
- Obermeyer Z., Powers B., Vogeli C., Mullainathan S. (2019) Dissecting racial bias in an algorithm used to manage the health of populations. *Science* 366:447–453.
- Geirhos R. *et al.* (2020) Shortcut learning in deep neural networks. *Nat Mach Intell* 2:665–673.
- Saerens M., Latinne P., Decaestecker C. (2002) Adjusting the outputs of a classifier to new a priori probabilities: a simple procedure. *Neural Comput* 14:21–41.
- Lipton Z. C., Wang Y.-X., Smola A. J. (2018) Detecting and correcting for label shift with black box predictors. *ICML*.
- Rabanser S., Günnemann S., Lipton Z. C. (2019) Failing loudly: an empirical study of methods for detecting dataset shift. *NeurIPS*.
- Platt J. C. (1999) Probabilistic outputs for support vector machines and comparisons to regularized likelihood methods. In *Advances in Large Margin Classifiers*, MIT Press.
- Guo C., Pleiss G., Sun Y., Weinberger K. Q. (2017) On calibration of modern neural networks. *ICML*.
- Gal Y., Ghahramani Z. (2016) Dropout as a Bayesian approximation: representing model uncertainty in deep learning. *ICML*.
- Lakshminarayanan B., Pritzel A., Blundell C. (2017) Simple and scalable predictive uncertainty estimation using deep ensembles. *NeurIPS*.
- Kendall A., Gal Y. (2017) What uncertainties do we need in Bayesian deep learning for computer vision? *NeurIPS*.
- Ovadia Y. *et al.* (2019) Can you trust your model's uncertainty? Evaluating predictive uncertainty under dataset shift. *NeurIPS*.
- Free S. M., Wilson J. W. (1964) A mathematical contribution to structure–activity studies. *J Med Chem* 7:395–399.
- Vovk V., Gammerman A., Shafer G. (2005) *Algorithmic Learning in a Random World*. Springer.
- Vovk V. (2012) Conditional validity of inductive conformal predictors. *Proc Asian Conf Mach Learn* (PMLR 25):475–490.
- Lei J., G'Sell M., Rinaldo A., Tibshirani R. J., Wasserman L. (2018) Distribution-free predictive inference for regression. *J Am Stat Assoc* 113:1094–1111.
- Romano Y., Patterson E., Candès E. J. (2019) Conformalized quantile regression. *NeurIPS*.
- Romano Y., Sesia M., Candès E. J. (2020) Classification with valid and adaptive coverage. *NeurIPS*.
- Tibshirani R. J., Barber R. F., Candès E. J., Ramdas A. (2019) Conformal prediction under covariate shift. *NeurIPS*.
- Angelopoulos A. N., Bates S. (2023) Conformal prediction: a gentle introduction. *Found Trends Mach Learn* 16:494–591.
- Fannjiang C., Bates S., Angelopoulos A. N., Listgarten J., Jordan M. I. (2022) Conformal prediction under feedback covariate shift for biomolecular design. *PNAS* 119:e2204569119.
- Strobl C., Boulesteix A.-L., Kneib T., Augustin T., Zeileis A. (2008) Conditional variable importance for random forests. *BMC Bioinformatics* 9:307.
- Adebayo J. *et al.* (2018) Sanity checks for saliency maps. *NeurIPS*.
- Martin A. R. *et al.* (2019) Clinical use of current polygenic risk scores may exacerbate health disparities. *Nat Genet* 51:584–591.
- Inker L. A. *et al.* (2021) New creatinine- and cystatin C–based equations to estimate GFR without race. *N Engl J Med* 385:1737–1749.
- Mitchell M. *et al.* (2019) Model cards for model reporting. *Proc Conf Fairness, Accountability, and Transparency (FAT\*)*.
- Gebru T. *et al.* (2021) Datasheets for datasets. *Commun ACM* 64:86–92.

## ➡️ Next

We have learned to distrust our own numbers in a productive way. The same scepticism is needed for the newest tools of
all: **[Module 43 · Foundation models, language models and AI agents for biology](43_llms_ai_agents.ipynb)** opens up
large language models, retrieval, tool-using agents and biological foundation models — and asks how to evaluate and use
them responsibly.